# `log_20260424_153546.csv` Recirculation Review

This notebook reviews the latest recirculation log with the April 24 cold run. The default flow-log density gate is widened here because the HFE density rises above `1600 kg/m3` during the coldest part of this run; keeping the old bound would misclassify the useful cold circulation as gas-rich.

The main derived thermal quantities use the calibrated thermocouple mean as the bulk-temperature proxy when available. That matters here: the flow-meter bottoms near `-94 C`, while the calibrated TC mean reaches roughly `-100 C` during the 40 percent cold hold.


In [1]:
from pathlib import Path
import importlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display
from scipy.interpolate import PchipInterpolator
from scipy.special import k0

import orca
import orca.logbook as orca_logbook

warnings.filterwarnings(
    'ignore',
    message='FigureCanvasAgg is non-interactive, and thus cannot be shown',
    category=UserWarning,
)

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('load_ext', 'autoreload')
    except Exception:
        pass
    ip.run_line_magic('autoreload', '2')

orca = importlib.reload(orca)
orca_logbook = importlib.reload(orca_logbook)

NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH
for candidate in [NB_PATH, *NB_PATH.parents]:
    if (candidate / 'data').exists() and (candidate / 'analysis').exists():
        REPO_ROOT = candidate
        break

LOG_PATH = REPO_ROOT / 'data' / 'raw' / 'recirculation' / 'log_20260424_153546.csv'
TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'
DENSITY_BOUNDS_KG_M3 = (1200.0, 1800.0)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'Repo root: {REPO_ROOT}')
print(f'Log path: {LOG_PATH}')
print(f'TC calibration: {TC_CALIBRATION_PATH}')
print(f'Density bounds: {DENSITY_BOUNDS_KG_M3[0]:.0f}-{DENSITY_BOUNDS_KG_M3[1]:.0f} kg/m3')


Repo root: /home/aamy/Documents/hfe-system
Log path: /home/aamy/Documents/hfe-system/data/raw/recirculation/log_20260424_153546.csv
TC calibration: /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv
Density bounds: 1200-1800 kg/m3


## Load and Segment the Log

The segment table uses the widened HFE liquid-density bounds. A command-step table is also built from the longest pump-running segment so later cells can refer to the cold 40 to 60 percent ramp without hard-coding the whole log by hand.


In [2]:
review = orca.prepare_flow_log_review(LOG_PATH, density_bounds=DENSITY_BOUNDS_KG_M3, tc_calibration_path=TC_CALIBRATION_PATH)

data = review.data.copy().sort_values('time_s').reset_index(drop=True)
display(Markdown(
    f'Applied restored-log thermocouple calibration in memory from `{TC_CALIBRATION_PATH.name}`. '
    'Raw thermocouple columns are retained as `*_raw_C` when calibration is applied.'
))
if review.valid_temp_cols:
    data['temp_mean_C'] = data[list(review.valid_temp_cols)].mean(axis=1)
    data['temp_min_C'] = data[list(review.valid_temp_cols)].min(axis=1)
    data['temp_max_C'] = data[list(review.valid_temp_cols)].max(axis=1)
    data['temp_span_C'] = data['temp_max_C'] - data['temp_min_C']
else:
    data['temp_mean_C'] = data['temperature_c_si']
    data['temp_min_C'] = data['temperature_c_si']
    data['temp_max_C'] = data['temperature_c_si']
    data['temp_span_C'] = np.nan

data['bulk_C'] = data['temp_mean_C'].where(data['temp_mean_C'].notna(), data['temperature_c_si'])
data['elapsed_from_log_start_min'] = (data['time_s'] - float(data['time_s'].iloc[0])) / 60.0

segment_summary = orca.build_segment_summary(data)
run_segment_id = review.run_segment_id if review.run_segment_id in segment_summary.index else int(segment_summary['use_for_quantitative_flow'].idxmax())
run = orca.segment_slice(data, segment_summary.loc[run_segment_id])
run_liquid, step_windows, step_summary, settle_cutoff_s = orca.command_step_summary(run)
step_windows = step_windows.copy()
step_windows['start_min'] = step_windows['start_s'] / 60.0
step_windows['end_min'] = step_windows['end_s'] / 60.0
step_windows['duration_min'] = step_windows['duration_s'] / 60.0

bypass_transition_mask = step_windows['cmd_pct'].shift(1).round().eq(40.0) & step_windows['cmd_pct'].round().eq(30.0)
if not bypass_transition_mask.any():
    raise ValueError('Could not find a 40% -> 30% pump-command transition for the bypass-closed marker.')
BYPASS_CLOSED_STEP_ID = int(step_windows.index[bypass_transition_mask][0])
BYPASS_CLOSED_TIME_S = float(step_windows.loc[BYPASS_CLOSED_STEP_ID, 'start_s'])
BYPASS_CLOSED_TIME_MIN = BYPASS_CLOSED_TIME_S / 60.0

step_detail_rows = []
for step_id, row in step_windows.iterrows():
    subset = data[
        data['time_s'].between(float(row['start_s']) + settle_cutoff_s, float(row['end_s']))
        & data['pump_running']
    ].copy()
    if subset.empty:
        subset = data[data['time_s'].between(float(row['start_s']), float(row['end_s']))].copy()
    step_detail_rows.append({
        'step_id': int(step_id),
        'cmd_pct': float(row['cmd_pct']),
        'start_min': float(row['start_min']),
        'end_min': float(row['end_min']),
        'duration_s': float(row['duration_s']),
        'samples': int(len(subset)),
        'bulk_median_C': float(subset['bulk_C'].median()),
        'bulk_min_C': float(subset['bulk_C'].min()),
        'flow_meter_median_C': float(subset['temperature_c_si'].median()),
        'volume_flow_lmin': float(subset['volume_flow_lmin_si'].median()),
        'mass_flow_kgmin': float(subset['mass_flow_kgmin_si'].median()),
        'delta_p_bar': float(subset['delta_p_bar_recomputed'].median()),
        'pump_power_w': float(subset['pump_input_power_w'].median()),
    })
step_detail = pd.DataFrame(step_detail_rows)

display(orca.flow_log_overview_table(review).to_frame())
display(segment_summary.round(3))
display(step_detail.round(3))

min_flow_c = float(data.loc[data['pump_running'], 'temperature_c_si'].min())
min_bulk_c = float(data.loc[data['pump_running'], 'bulk_C'].min())
display(Markdown(
    f'Flow-meter minimum while pumping: `{min_flow_c:.1f} C`; '
    f'TC-mean bulk proxy minimum while pumping: `{min_bulk_c:.1f} C`. '
    f'Bypass-closed marker from the first `40% -> 30%` command transition: '
    f'`{BYPASS_CLOSED_TIME_MIN:.2f} min`.'
))


Applied restored-log thermocouple calibration in memory from `TC_calibration_20260420.csv`. Raw thermocouple columns are retained as `*_raw_C` when calibration is applied.

,value
flow_note,"This CSV already looks SI-like, so the noteboo..."
rows,9467
time_start_s,1220.483
time_end_s,21023.023
run_segment_id,1
dominant_command_pct,40.0
stable_hold_start_s,2975.482
stable_hold_end_s,6431.62
stable_hold_duration_s,3456.138
valid_thermocouples,"TFO, TTI, TTO, TMI, THI, THM"


,start_s,end_s,duration_s,cmd_level_count,pump_cmd_levels_pct,median_freq_hz,max_freq_hz,median_mass_flow_kgmin,median_volume_flow_lmin,median_density_kg_m3,median_flow_temp_C,liquid_fraction,positive_mass_flow_fraction,flow_temp_change_C,density_change_kg_m3,classification,use_for_quantitative_flow
segment_id,,,,,,,,,,,,,,,,,
1,1220.483,17607.292,16386.809,7,"20, 30, 40, 45, 50, 55, 60",28.14,42.22,4.33,2.686,1614.0,-65.99,1.0,1.0,-48.591,102.0,usable liquid circulation,True


,step_id,cmd_pct,start_min,end_min,duration_s,samples,bulk_median_C,bulk_min_C,flow_meter_median_C,volume_flow_lmin,mass_flow_kgmin,delta_p_bar,pump_power_w
0,0,20.0,20.341,24.316,238.461,114,20.040,-0.441,21.525,1.298,1.866,0.043,60.0
1,1,30.0,24.351,48.685,1460.053,697,-50.999,-68.784,-36.302,1.976,3.077,0.177,63.0
2,2,40.0,48.720,57.924,552.263,264,-69.990,-75.794,-56.896,2.696,4.309,0.392,66.0
3,3,30.0,57.959,62.003,242.647,116,-75.493,-79.668,-61.720,2.011,3.235,0.371,63.0
4,4,40.0,62.038,101.265,2353.604,1125,-88.351,-95.305,-76.195,2.704,4.427,0.704,68.0
5,5,45.0,101.300,104.474,190.428,91,-94.852,-95.960,-80.916,3.055,5.028,0.987,72.0
6,6,40.0,104.509,167.344,3770.095,1802,-99.347,-102.737,-90.672,2.736,4.557,1.055,71.0
7,7,45.0,167.378,167.797,25.101,12,-99.049,-99.162,-93.932,3.086,5.159,1.423,76.0
8,8,50.0,167.832,168.948,66.948,31,-98.503,-98.854,-94.167,3.424,5.725,1.691,80.0
9,9,55.0,168.983,169.575,35.559,17,-96.774,-97.351,-93.366,3.746,6.254,1.887,84.0


Flow-meter minimum while pumping: `-94.4 C`; TC-mean bulk proxy minimum while pumping: `-102.7 C`. Bypass-closed marker from the first `40% -> 30%` command transition: `57.96 min`.

## Warmup Thermocouple Residuals and Stratification

This diagnostic starts at the coldest mixed-loop reference temperature and keeps only the following warmup. The reference is the mean of the well-wetted loop probes, `TFO`, `TTI`, `TTO`, and `TMI`; each connected thermocouple is plotted as `TC - loop_reference_mean` versus elapsed time. The secondary axis shows loop-probe span and all-TC span so the low-stratification transfer-calibration window is visible directly on the residual plot.


In [3]:
WARMUP_REFERENCE_COLS = ['TFO_C', 'TTI_C', 'TTO_C', 'TMI_C']
WARMUP_LOW_LOOP_SPAN_C = 1.0
WARMUP_LOW_ALL_TC_SPAN_C = 3.0
WARMUP_MIN_LOW_WINDOW_MIN = 2.0
WARMUP_SAMPLE_STRIDE = 8
WARMUP_ROLLING_SAMPLES = 61

available_ref_cols = [
    col for col in WARMUP_REFERENCE_COLS
    if col in data.columns and col in review.valid_temp_cols
]
if len(available_ref_cols) < 3:
    raise RuntimeError(
        f'Need at least three loop reference probes for the warmup diagnostic; found {available_ref_cols}.'
    )

tc_residual_cols = [col for col in review.valid_temp_cols if col in data.columns]
if not tc_residual_cols:
    raise RuntimeError('No connected thermocouple columns found for residual plotting.')

elapsed_col = 'elapsed_from_log_start_min'
if elapsed_col not in data.columns:
    data[elapsed_col] = (data['time_s'] - float(data['time_s'].iloc[0])) / 60.0

plot_cols = [elapsed_col, 'time_s', 'pump_cmd_pct', 'pump_running', *tc_residual_cols]
plot_cols = [col for col in dict.fromkeys(plot_cols) if col in data.columns]
warmup_frame = data[plot_cols].copy()
for col in [elapsed_col, *tc_residual_cols]:
    warmup_frame[col] = pd.to_numeric(warmup_frame[col], errors='coerce')

warmup_frame['loop_reference_mean_C'] = warmup_frame[available_ref_cols].mean(axis=1)
warmup_frame['loop_reference_span_C'] = (
    warmup_frame[available_ref_cols].max(axis=1) - warmup_frame[available_ref_cols].min(axis=1)
)
warmup_frame['all_tc_span_C'] = (
    warmup_frame[tc_residual_cols].max(axis=1) - warmup_frame[tc_residual_cols].min(axis=1)
)

warmup_frame = warmup_frame.dropna(subset=[elapsed_col, 'loop_reference_mean_C']).sort_values(elapsed_col)
warmup_start_idx = warmup_frame['loop_reference_mean_C'].idxmin()
warmup_start_min = float(warmup_frame.loc[warmup_start_idx, elapsed_col])
warmup_frame = warmup_frame.loc[warmup_frame[elapsed_col].ge(warmup_start_min)].reset_index(drop=True)
if warmup_frame.empty:
    raise RuntimeError('No warmup samples found after the coldest loop-reference point.')

warmup_frame['low_loop_stratification'] = warmup_frame['loop_reference_span_C'].le(WARMUP_LOW_LOOP_SPAN_C)
warmup_frame['low_all_tc_stratification'] = warmup_frame['all_tc_span_C'].le(WARMUP_LOW_ALL_TC_SPAN_C)

warmup_long_rows = []
for col in tc_residual_cols:
    work = warmup_frame[[elapsed_col, 'loop_reference_mean_C', 'loop_reference_span_C', 'all_tc_span_C']].copy()
    work['tc'] = col.replace('_C', '')
    work['residual_C'] = warmup_frame[col] - warmup_frame['loop_reference_mean_C']
    warmup_long_rows.append(work)
warmup_residual_long = pd.concat(warmup_long_rows, ignore_index=True).dropna(subset=['residual_C'])

low_window_rows = []
low_mask = warmup_frame['low_all_tc_stratification'].fillna(False)
window_ids = low_mask.ne(low_mask.shift(fill_value=False)).cumsum()
for _, group in warmup_frame.loc[low_mask].groupby(window_ids[low_mask]):
    start_min = float(group[elapsed_col].iloc[0])
    end_min = float(group[elapsed_col].iloc[-1])
    duration_min = end_min - start_min
    if duration_min < WARMUP_MIN_LOW_WINDOW_MIN:
        continue
    row = {
        'start_min': start_min,
        'end_min': end_min,
        'duration_min': duration_min,
        'samples': int(len(group)),
        'loop_ref_median_C': float(group['loop_reference_mean_C'].median()),
        'loop_span_median_C': float(group['loop_reference_span_C'].median()),
        'all_tc_span_median_C': float(group['all_tc_span_C'].median()),
    }
    if 'pump_cmd_pct' in group.columns:
        row['pump_cmd_median_pct'] = float(group['pump_cmd_pct'].median())
    if 'pump_running' in group.columns:
        row['pump_running_fraction'] = float(group['pump_running'].mean())
    for probe in ['THI_C', 'THM_C']:
        if probe in group.columns:
            row[f'{probe.replace("_C", "")}_minus_ref_median_C'] = float(
                (group[probe] - group['loop_reference_mean_C']).median()
            )
    low_window_rows.append(row)
low_stratification_windows = pd.DataFrame(low_window_rows)

fig, ax = plt.subplots(figsize=(11.0, 6.3), constrained_layout=True)
color_cycle = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
color_by_tc = {
    tc: color_cycle[index % len(color_cycle)]
    for index, tc in enumerate(warmup_residual_long['tc'].unique())
}

if not low_stratification_windows.empty:
    for index, row in low_stratification_windows.iterrows():
        ax.axvspan(
            row['start_min'],
            row['end_min'],
            color='0.82',
            alpha=0.22,
            linewidth=0,
            label=f'all TC span <= {WARMUP_LOW_ALL_TC_SPAN_C:.1f} C' if index == 0 else None,
        )

for tc, group in warmup_residual_long.groupby('tc', sort=False):
    group = group.sort_values(elapsed_col).reset_index(drop=True)
    sample = group.iloc[::WARMUP_SAMPLE_STRIDE]
    ax.scatter(
        sample[elapsed_col],
        sample['residual_C'],
        s=8,
        alpha=0.10,
        color=color_by_tc[tc],
        linewidths=0,
    )
    smooth = group['residual_C'].rolling(
        WARMUP_ROLLING_SAMPLES,
        center=True,
        min_periods=max(5, WARMUP_ROLLING_SAMPLES // 6),
    ).median()
    ax.plot(
        group[elapsed_col],
        smooth,
        lw=1.6,
        color=color_by_tc[tc],
        label=tc,
    )

ax.axhline(0.0, color='0.25', lw=1.0, ls='--', label='loop reference mean')
ax.axvline(warmup_start_min, color='0.35', lw=1.0, ls=':', label='warmup start')
if 'pump_cmd_pct' in warmup_frame.columns:
    pump_cmd = pd.to_numeric(warmup_frame['pump_cmd_pct'], errors='coerce').fillna(0.0)
    pump_off_times = warmup_frame.loc[
        pump_cmd.le(0.0) & pump_cmd.shift(fill_value=float(pump_cmd.iloc[0])).gt(0.0),
        elapsed_col,
    ]
    for index, pump_off_min in enumerate(pump_off_times):
        ax.axvline(
            float(pump_off_min),
            color='0.45',
            lw=1.0,
            ls='-.',
            alpha=0.85,
            label='pump off' if index == 0 else None,
        )
ax.set_xlabel('Elapsed from log start [min]')
ax.set_ylabel('TC - loop reference mean [C]')
ax.set_title('Warmup thermocouple residuals and stratification')
ax.grid(True, alpha=0.25)

ax2 = ax.twinx()
ax2.plot(
    warmup_frame[elapsed_col],
    warmup_frame['loop_reference_span_C'],
    color='0.10',
    lw=1.3,
    alpha=0.80,
    label='loop probe span',
)
ax2.plot(
    warmup_frame[elapsed_col],
    warmup_frame['all_tc_span_C'],
    color='0.10',
    lw=1.2,
    alpha=0.65,
    ls=':',
    label='all TC span',
)
ax2.axhline(WARMUP_LOW_LOOP_SPAN_C, color='0.35', lw=0.9, ls='--', alpha=0.75)
ax2.axhline(WARMUP_LOW_ALL_TC_SPAN_C, color='0.35', lw=0.9, ls='-.', alpha=0.75)
ax2.set_ylabel('Temperature span [C]')
ax2.grid(False)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles + handles2, labels + labels2, loc='upper left', ncol=2, fontsize=8)

plt.show()

if low_stratification_windows.empty:
    display(Markdown(
        f'No warmup interval stayed below all-TC span `{WARMUP_LOW_ALL_TC_SPAN_C:.1f} C` '
        f'for at least `{WARMUP_MIN_LOW_WINDOW_MIN:.1f} min`.'
    ))
else:
    display(
        low_stratification_windows
        .style.format({
            'start_min': '{:.2f}',
            'end_min': '{:.2f}',
            'duration_min': '{:.2f}',
            'samples': '{:.0f}',
            'loop_ref_median_C': '{:.2f}',
            'loop_span_median_C': '{:.2f}',
            'all_tc_span_median_C': '{:.2f}',
            'pump_cmd_median_pct': '{:.0f}',
            'pump_running_fraction': '{:.2f}',
            'THI_minus_ref_median_C': '{:+.2f}',
            'THM_minus_ref_median_C': '{:+.2f}',
        })
        .hide(axis='index')
    )


start_min,end_min,duration_min,samples,loop_ref_median_C,loop_span_median_C,all_tc_span_median_C,pump_cmd_median_pct,pump_running_fraction,THI_minus_ref_median_C,THM_minus_ref_median_C
163.91,274.72,110.80,3179,-48.02,0.52,0.91,40,0.99,+0.44,-0.45


## Warmup TTI Minus TTO

This plot isolates the pump-on warmup after cooldown stops. The start is five minutes after the final cooldown-valve falling edge, and the end is the first pump-off command. The plotted quantity is `TTI - TTO`, which is a direct inlet/outlet stratification proxy for the loop during warmup.


In [4]:
WARMUP_AFTER_COOLDOWN_DELAY_MIN = 5.0
DELTA_ROLLING_SAMPLES = 61

required_cols = ['TTI_C', 'TTO_C', 'valve', 'pump_cmd_pct']
missing_cols = [col for col in required_cols if col not in data.columns]
if missing_cols:
    raise RuntimeError(f'Missing required columns for warmup TTI-TTO plot: {missing_cols}')

elapsed_col = 'elapsed_from_log_start_min'
if elapsed_col not in data.columns:
    data[elapsed_col] = (data['time_s'] - float(data['time_s'].iloc[0])) / 60.0

warmup_delta_source = data[[elapsed_col, 'time_s', 'TTI_C', 'TTO_C', 'valve', 'pump_cmd_pct']].copy()
for col in [elapsed_col, 'time_s', 'TTI_C', 'TTO_C', 'valve', 'pump_cmd_pct']:
    warmup_delta_source[col] = pd.to_numeric(warmup_delta_source[col], errors='coerce')
warmup_delta_source = warmup_delta_source.dropna(subset=[elapsed_col, 'TTI_C', 'TTO_C']).sort_values(elapsed_col)

valve_state = warmup_delta_source['valve'].fillna(0.0)
pump_cmd = warmup_delta_source['pump_cmd_pct'].fillna(0.0)

pump_off_candidates = warmup_delta_source.loc[
    pump_cmd.le(0.0) & pump_cmd.shift(fill_value=float(pump_cmd.iloc[0])).gt(0.0),
    elapsed_col,
]
if pump_off_candidates.empty:
    raise RuntimeError('Could not find a pump-off command transition.')
pump_off_min = float(pump_off_candidates.iloc[0])

valve_falling_edges = warmup_delta_source.loc[
    valve_state.le(0.0) & valve_state.shift(fill_value=float(valve_state.iloc[0])).gt(0.0),
    elapsed_col,
]
valve_falling_edges = valve_falling_edges[valve_falling_edges.lt(pump_off_min)]
if valve_falling_edges.empty:
    raise RuntimeError('Could not find a cooldown-valve falling edge before pump-off.')
cooldown_stopped_min = float(valve_falling_edges.iloc[-1])
plot_start_min = cooldown_stopped_min + WARMUP_AFTER_COOLDOWN_DELAY_MIN
plot_end_min = pump_off_min

warmup_delta_frame = warmup_delta_source.loc[
    warmup_delta_source[elapsed_col].ge(plot_start_min)
    & warmup_delta_source[elapsed_col].lt(plot_end_min)
].copy()
if warmup_delta_frame.empty:
    raise RuntimeError(
        'No warmup samples found between five minutes after cooldown stop and pump-off.'
    )

warmup_delta_frame['minutes_after_cooldown_stop'] = (
    warmup_delta_frame[elapsed_col] - cooldown_stopped_min
)
warmup_delta_frame['TTI_minus_TTO_C'] = warmup_delta_frame['TTI_C'] - warmup_delta_frame['TTO_C']
warmup_delta_frame['TTI_minus_TTO_rolling_C'] = warmup_delta_frame['TTI_minus_TTO_C'].rolling(
    DELTA_ROLLING_SAMPLES,
    center=True,
    min_periods=max(5, DELTA_ROLLING_SAMPLES // 6),
).median()
warmup_delta_frame['TTI_rolling_C'] = warmup_delta_frame['TTI_C'].rolling(
    DELTA_ROLLING_SAMPLES,
    center=True,
    min_periods=max(5, DELTA_ROLLING_SAMPLES // 6),
).median()

fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
ax.plot(
    warmup_delta_frame['minutes_after_cooldown_stop'],
    warmup_delta_frame['TTI_minus_TTO_C'],
    color='0.55',
    lw=0.8,
    alpha=0.45,
    label='sample',
)
ax.plot(
    warmup_delta_frame['minutes_after_cooldown_stop'],
    warmup_delta_frame['TTI_minus_TTO_rolling_C'],
    color='tab:blue',
    lw=2.0,
    label='rolling median',
)
ax.axhline(0.0, color='0.20', lw=1.0, ls='--')
ax.axhline(
    warmup_delta_frame['TTI_minus_TTO_C'].median(),
    color='tab:orange',
    lw=1.2,
    ls=':',
    label='delta window median',
)
ax.axvline(WARMUP_AFTER_COOLDOWN_DELAY_MIN, color='0.35', lw=1.0, ls=':', label='+5 min after cooldown stop')
ax.axvline(plot_end_min - cooldown_stopped_min, color='0.35', lw=1.0, ls='-.', label='pump off')
ax.set_xlabel('Minutes after cooldown stopped [min]')
ax.set_ylabel('TTI - TTO [C]')
ax.set_title('Warmup inlet-outlet temperature difference and TTI before pump shutdown')
ax.grid(True, alpha=0.25)

ax2 = ax.twinx()
ax2.plot(
    warmup_delta_frame['minutes_after_cooldown_stop'],
    warmup_delta_frame['TTI_C'],
    color='tab:red',
    lw=0.8,
    alpha=0.18,
    label='TTI sample',
)
ax2.plot(
    warmup_delta_frame['minutes_after_cooldown_stop'],
    warmup_delta_frame['TTI_rolling_C'],
    color='tab:red',
    lw=2.0,
    alpha=0.90,
    label='TTI rolling median',
)
ax2.set_ylabel('TTI [C]')
ax2.grid(False)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles + handles2, labels + labels2, loc='best', fontsize=8)
plt.show()

warmup_tti_tto_summary = pd.DataFrame([{
    'cooldown_stopped_min': cooldown_stopped_min,
    'plot_start_min': plot_start_min,
    'pump_off_min': pump_off_min,
    'duration_min': float(warmup_delta_frame[elapsed_col].max() - warmup_delta_frame[elapsed_col].min()),
    'samples': int(len(warmup_delta_frame)),
    'TTI_minus_TTO_mean_C': float(warmup_delta_frame['TTI_minus_TTO_C'].mean()),
    'TTI_minus_TTO_median_C': float(warmup_delta_frame['TTI_minus_TTO_C'].median()),
    'TTI_minus_TTO_p10_C': float(warmup_delta_frame['TTI_minus_TTO_C'].quantile(0.10)),
    'TTI_minus_TTO_p90_C': float(warmup_delta_frame['TTI_minus_TTO_C'].quantile(0.90)),
    'TTI_start_C': float(warmup_delta_frame['TTI_C'].iloc[0]),
    'TTI_end_C': float(warmup_delta_frame['TTI_C'].iloc[-1]),
    'TTI_median_C': float(warmup_delta_frame['TTI_C'].median()),
}])

display(
    warmup_tti_tto_summary
    .style.format({
        'cooldown_stopped_min': '{:.2f}',
        'plot_start_min': '{:.2f}',
        'pump_off_min': '{:.2f}',
        'duration_min': '{:.2f}',
        'samples': '{:.0f}',
        'TTI_minus_TTO_mean_C': '{:+.3f}',
        'TTI_minus_TTO_median_C': '{:+.3f}',
        'TTI_minus_TTO_p10_C': '{:+.3f}',
        'TTI_minus_TTO_p90_C': '{:+.3f}',
        'TTI_start_C': '{:.2f}',
        'TTI_end_C': '{:.2f}',
        'TTI_median_C': '{:.2f}',
    })
    .hide(axis='index')
)


cooldown_stopped_min,plot_start_min,pump_off_min,duration_min,samples,TTI_minus_TTO_mean_C,TTI_minus_TTO_median_C,TTI_minus_TTO_p10_C,TTI_minus_TTO_p90_C,TTI_start_C,TTI_end_C,TTI_median_C
174.13,179.13,273.08,93.89,2694,+0.273,+0.230,-0.030,+0.660,-68.35,-27.77,-44.90


## General Plot With LN Weight

The LN trace uses the scale gross weight and the configured tare. The net LN estimate below is `scale_weight_kg - median_positive_tare`, then lightly smoothed so the stepwise scale readout is readable on the same time axis.


In [5]:
def _apply_fine_grid(ax):
    ax.minorticks_on()
    ax.grid(True, which='major', alpha=0.34, linewidth=0.8)
    ax.grid(True, which='minor', alpha=0.14, linestyle=':', linewidth=0.6)


def _prepare_ln_weight_columns(frame):
    work = frame.copy()
    positive_tare = pd.to_numeric(work['scale_tare_kg'], errors='coerce')
    positive_tare = positive_tare[positive_tare > 0]
    tare_ref_kg = float(positive_tare.median()) if not positive_tare.empty else 0.0
    work['ln_gross_weight_kg'] = pd.to_numeric(work['scale_weight_kg'], errors='coerce')
    work['ln_net_weight_kg'] = work['ln_gross_weight_kg'] - tare_ref_kg
    valid = work['scale_age_s'].le(2.0) & work['ln_net_weight_kg'].between(-5.0, 80.0)
    work.loc[~valid, 'ln_net_weight_kg'] = np.nan
    work['ln_net_weight_smooth_kg'] = (
        work['ln_net_weight_kg']
        .rolling(31, center=True, min_periods=5)
        .median()
        .interpolate(limit_direction='both')
    )
    return work, tare_ref_kg


data, scale_tare_ref_kg = _prepare_ln_weight_columns(data)


def plot_general_overview_with_ln_weight(frame):
    fig, axes = plt.subplots(5, 1, figsize=(15, 14), sharex=True, constrained_layout=True)
    t_min = frame['t_min']

    ax_cmd = axes[0]
    ax_freq = ax_cmd.twinx()
    ax_cmd.plot(t_min, frame['pump_cmd_pct'], color='#111827', lw=1.4, label='Pump command [%]')
    ax_freq.plot(t_min, frame['pump_freq_hz'], color='#6d28d9', lw=1.05, alpha=0.9, label='Pump frequency [Hz]')
    ax_cmd.fill_between(t_min, 0, 100, where=frame['valve'].fillna(0).ge(0.5), color='#60a5fa', alpha=0.12, step='mid', label='LN valve open')
    ax_cmd.set_ylabel('Command [%]', fontsize=12)
    ax_cmd.set_ylim(0, 100)
    ax_freq.set_ylabel('Frequency [Hz]', fontsize=12)
    ax_freq.set_ylim(0, max(72.0, 1.08 * float(frame['pump_freq_hz'].max())))
    _apply_fine_grid(ax_cmd)
    _apply_fine_grid(ax_freq)
    lines = ax_cmd.get_lines() + ax_freq.get_lines()
    labels = [line.get_label() for line in lines]
    handles = lines + [
        plt.Line2D([0], [0], color='#60a5fa', lw=8, alpha=0.25),
        plt.Line2D([0], [0], color='#be123c', lw=1.5, ls='--'),
    ]
    ax_cmd.legend(handles, labels + ['LN valve open', 'Bypass closed'], loc='upper right', fontsize=8)
    ax_cmd.set_title('April 24 recirculation overview with LN scale weight')

    ax_temp = axes[1]
    temp_sensor_cols = [column for column in review.valid_temp_cols if column in frame.columns]
    sensor_colors = plt.cm.tab20(np.linspace(0.0, 1.0, max(len(temp_sensor_cols), 1)))
    for color, column in zip(sensor_colors, temp_sensor_cols):
        ax_temp.plot(
            t_min,
            frame[column],
            lw=0.85,
            alpha=0.72,
            color=color,
            label=column.replace('_C', ''),
        )
    ax_temp.plot(t_min, frame['temperature_c_si'], lw=1.2, color='#111827', label='Flow-meter temp [C]')
    ax_temp.plot(t_min, frame['bulk_C'], lw=1.7, color='#dc2626', label='TC mean bulk proxy [C]')
    ax_temp.fill_between(t_min, frame['temp_min_C'], frame['temp_max_C'], color='#f97316', alpha=0.08, label='TC min-max')
    ax_temp.set_ylabel('Temperature [C]', fontsize=12)
    ax_temp.legend(loc='best', fontsize=7, ncols=4)
    _apply_fine_grid(ax_temp)

    ax_flow = axes[2]
    ax_vol = ax_flow.twinx()
    ax_flow.plot(t_min, frame['mass_flow_kgmin_si'], color='#2563eb', lw=1.15, label='Mass flow [kg/min]')
    ax_vol.plot(t_min, frame['volume_flow_lmin_si'], color='#dc2626', lw=1.0, label='Volume flow [L/min]')
    ax_flow.set_ylabel('Mass flow [kg/min]', fontsize=12)
    ax_vol.set_ylabel('Volume flow [L/min]', fontsize=12)
    _apply_fine_grid(ax_flow)
    _apply_fine_grid(ax_vol)
    lines = ax_flow.get_lines() + ax_vol.get_lines()
    ax_flow.legend(lines, [line.get_label() for line in lines], loc='best', fontsize=8)

    ax_pressure = axes[3]
    ax_pressure.plot(t_min, frame['pump_pressure_before_bar_abs'], label='Before pump [bar abs]', lw=1.0)
    ax_pressure.plot(t_min, frame['pump_pressure_after_bar_abs'], label='After pump [bar abs]', lw=1.0)
    ax_pressure.plot(t_min, frame['pump_pressure_tank_bar_abs'], label='Tank [bar abs]', lw=1.0)
    ax_pressure.plot(t_min, frame['delta_p_bar_recomputed'], label='Pump pressure rise [bar]', lw=1.0, color='#111827')
    ax_pressure.set_ylabel('Pressure [bar]', fontsize=12)
    ax_pressure.legend(loc='best', fontsize=8, ncols=4)
    _apply_fine_grid(ax_pressure)

    ax_weight = axes[4]
    ax_weight.plot(t_min, frame['ln_net_weight_smooth_kg'], color='#0f766e', lw=1.8, label=f'Net LN weight, tare={scale_tare_ref_kg:.1f} kg')
    ax_weight.scatter(t_min, frame['ln_net_weight_kg'], s=5, color='#0f766e', alpha=0.18, label='Net LN samples')
    ax_weight.set_ylabel('Net LN weight [kg]', fontsize=12)
    ax_weight.set_xlabel('Elapsed log time [min]', fontsize=12)
    ax_weight.legend(loc='best', fontsize=8)
    _apply_fine_grid(ax_weight)

    for ax in axes:
        ax.axvline(BYPASS_CLOSED_TIME_MIN, color='#be123c', ls='--', lw=1.35, alpha=0.88)
        ax.set_xlim(float(t_min.min()), float(t_min.max()))
    ax_cmd.annotate(
        'Bypass closed\n40 -> 30%',
        xy=(BYPASS_CLOSED_TIME_MIN, 92),
        xytext=(7, -6),
        textcoords='offset points',
        ha='left',
        va='top',
        fontsize=8,
        color='#be123c',
        bbox=dict(facecolor='white', edgecolor='#be123c', alpha=0.88, boxstyle='round,pad=0.25'),
    )
    return fig

plot_general_overview_with_ln_weight(data)
plt.show()


## Warmup Heat Leaks

Two warmup intervals are fit with a simple ambient-return model. The pump-on warmup after the cold ramp is the preferred heat-leak calibration because the loop is still circulating and the TC mean stays spatially coherent. The pump-off tail is shown as a cautionary comparison; stratification grows quickly once circulation stops.


In [6]:
model = orca.default_system_model()
ambient_c = float(model.ambient_temp_k - 273.15)
active_hfe_liquid_kg = model.fluid_volume_m3 * orca.hfe_density_kg_m3(model.ambient_temp_k)
steel_heat_capacity_j_k = model.steel_mass_kg * model.steel_cp_j_kgk

# Step after the 60 percent cold ramp: the post-ramp 40 percent warmup.
sixty_steps = step_windows[step_windows['cmd_pct'].eq(60.0)]
if sixty_steps.empty:
    raise RuntimeError('Could not find the 60 percent cold-ramp step.')
sixty_step_id = int(sixty_steps.index[-1])
post_ramp_40 = step_windows[(step_windows.index > sixty_step_id) & step_windows['cmd_pct'].eq(40.0)].iloc[0]

pump_end_s = float(data.loc[data['pump_running'], 'time_s'].max())
warmup_segments = [
    {
        'label': 'W1',
        'description': 'pump on, 40 percent, valve closed',
        'color': '#2563eb',
        'temperature_col': 'bulk_C',
        'fit_start_min': 5.0,
        'frame': data[
            data['time_s'].between(float(post_ramp_40['start_s']), float(post_ramp_40['end_s']))
            & data['pump_running']
        ].copy(),
    },
    {
        'label': 'W2',
        'description': 'pump off tail, stratified check',
        'color': '#ea580c',
        'temperature_col': 'bulk_C',
        'fit_start_min': 2.0,
        'frame': data[(data['time_s'].gt(pump_end_s)) & (~data['pump_running'])].copy(),
    },
]

warmup_fit_rows = []
fig, ax = plt.subplots(figsize=(13, 6.5), constrained_layout=True)

for segment in warmup_segments:
    frame = segment['frame'].sort_values('time_s').reset_index(drop=True)
    frame['elapsed_min'] = (frame['time_s'] - float(frame['time_s'].iloc[0])) / 60.0
    if len(frame) < 20:
        continue
    fit = orca.fit_warmup_segment(
        frame,
        active_hfe_liquid_kg=active_hfe_liquid_kg,
        temperature_col=segment['temperature_col'],
        time_col='time_s',
        pump_power_col='pump_input_power_w',
        fit_start_min=segment['fit_start_min'],
        min_samples=20,
        model=model,
    )
    segment['fit'] = fit
    segment['frame'] = frame

    fit_elapsed_min = fit.fit_start_min + fit.fit_elapsed_s / 60.0
    curve_elapsed_min = np.linspace(fit.fit_start_min, fit.fit_end_min, 300)
    curve_temp_c = fit.predict_temperature_C((curve_elapsed_min - fit.fit_start_min) * 60.0)

    ax.plot(frame['elapsed_min'], frame[segment['temperature_col']], color=segment['color'], alpha=0.35, lw=1.1)
    ax.plot(fit_elapsed_min, fit.fit_temperature_C, color=segment['color'], alpha=0.75, lw=1.25)
    ax.plot(curve_elapsed_min, curve_temp_c, color=segment['color'], lw=2.2, ls='--', label=f"{segment['label']} - {segment['description']}")
    ax.axvline(segment['fit_start_min'], color=segment['color'], lw=1.0, ls=':', alpha=0.9)

    warmup_fit_rows.append({
        'segment': segment['label'],
        'description': segment['description'],
        'samples': fit.n_samples,
        'fit_start_min': fit.fit_start_min,
        'fit_end_min': fit.fit_end_min,
        'initial_fit_temp_C': fit.initial_temp_c,
        'ua_w_per_k': fit.ua_w_per_k,
        'ua_sigma_w_per_k': fit.ua_sigma_w_per_k,
        'tau_min': fit.tau_s / 60.0,
        'r2': fit.r_squared,
        'rmse_C': fit.rmse_C,
        'heat_leak_at_minus_100_w': fit.heat_leak_at_temperature_C(-100.0),
        'heat_leak_at_minus_90_w': fit.heat_leak_at_temperature_C(-90.0),
        'heat_leak_at_minus_70_w': fit.heat_leak_at_temperature_C(-70.0),
        'heat_leak_at_minus_50_w': fit.heat_leak_at_temperature_C(-50.0),
    })

ax.set_xlabel('Elapsed warmup time [min]')
ax.set_ylabel('Bulk temperature proxy [C]')
ax.set_title('Warmup heat-leak fits')
ax.legend(loc='best', fontsize=9)
_apply_fine_grid(ax)
plt.show()

warmup_fit_summary = pd.DataFrame(warmup_fit_rows).set_index('segment')
display(warmup_fit_summary.round(3))

preferred_warmup = next(segment for segment in warmup_segments if segment['label'] == 'W1')
preferred_ua_w_per_k = float(preferred_warmup['fit'].ua_w_per_k)
display(Markdown(
    f'Using `W1` for later cooling-power estimates: `UA = {preferred_ua_w_per_k:.3f} W/K`. '
    f'Active HFE mass from the repository model is `{active_hfe_liquid_kg:.2f} kg`; '
    f'steel heat capacity is `{steel_heat_capacity_j_k:.0f} J/K`.'
))


,description,samples,fit_start_min,fit_end_min,initial_fit_temp_C,ua_w_per_k,ua_sigma_w_per_k,tau_min,r2,rmse_C,heat_leak_at_minus_100_w,heat_leak_at_minus_90_w,heat_leak_at_minus_70_w,heat_leak_at_minus_50_w
segment,,,,,,,,,,,,,,
W1,"pump on, 40 percent, valve closed",3316,5.021,120.602,-84.922,1.069,0.047,142.686,0.997,0.87,128.126,117.435,96.054,74.673
W2,"pump off tail, stratified check",1575,2.022,56.894,-25.873,1.614,0.105,97.014,0.975,0.80,193.379,177.244,144.974,112.703


Using `W1` for later cooling-power estimates: `UA = 1.069 W/K`. Active HFE mass from the repository model is `4.39 kg`; steel heat capacity is `4269 J/K`.

## Cooling Power and Warmup Heat-Leak History

`Q_inventory = -C_eff dT/dt` is positive while the system is cooling. The gross HX-removal estimate adds the ambient heat leak from the preferred warmup calibration. The LN latent-power trace is an independent scale-derived check using a rolling slope of the smoothed net LN weight; it is noisier because the scale reports in steps and has occasional spikes.


In [7]:
DERIVATIVE_WINDOW_S = 240.0
LN_DERIVATIVE_WINDOW_S = 300.0
LN_LATENT_HEAT_J_KG = orca.default_cooldown_design_inputs().ln2_latent_heat_j_kg

thermal = data.copy().sort_values('time_s').reset_index(drop=True)
for column in ['ln_net_weight_smooth_kg', 'ln_rate_kg_s', 'ln_latent_power_w']:
    if column in thermal.columns:
        thermal = thermal.drop(columns=column)
thermal['bulk_rate_C_s'] = orca.rolling_slope(
    thermal['time_s'].to_numpy(float),
    thermal['bulk_C'].to_numpy(float),
    window_s=DERIVATIVE_WINDOW_S,
    min_pts=11,
)
thermal['capacity_j_per_k'] = [
    orca.thermal_capacity_j_per_k(model, float(temp_c) + 273.15)
    for temp_c in thermal['bulk_C']
]
thermal['q_inventory_cooling_w'] = -thermal['capacity_j_per_k'] * thermal['bulk_rate_C_s']
thermal['q_leak_model_w'] = preferred_ua_w_per_k * np.clip(ambient_c - thermal['bulk_C'], 0.0, None)
thermal['q_hx_gross_w'] = np.clip(thermal['q_inventory_cooling_w'] + thermal['q_leak_model_w'], 0.0, None)
thermal['q_warmup_from_rate_w'] = np.clip(thermal['capacity_j_per_k'] * thermal['bulk_rate_C_s'], 0.0, None)

scale_valid = thermal['scale_age_s'].le(2.0)
scale = thermal.loc[scale_valid, ['time_s', 't_min', 'ln_net_weight_kg']].copy()
scale['ln_net_weight_smooth_kg'] = (
    scale['ln_net_weight_kg']
    .rolling(61, center=True, min_periods=10)
    .median()
    .interpolate(limit_direction='both')
)
scale['ln_rate_kg_s'] = orca.rolling_slope(
    scale['time_s'].to_numpy(float),
    scale['ln_net_weight_smooth_kg'].to_numpy(float),
    window_s=LN_DERIVATIVE_WINDOW_S,
    min_pts=20,
)
scale['ln_latent_power_w'] = np.clip(-scale['ln_rate_kg_s'] * LN_LATENT_HEAT_J_KG, 0.0, 1500.0)
thermal = thermal.merge(
    scale[['time_s', 'ln_net_weight_smooth_kg', 'ln_rate_kg_s', 'ln_latent_power_w']],
    on='time_s',
    how='left',
)

# Summarize only the cooling part of each command step.
power_rows = []
for step_id, row in step_windows.iterrows():
    subset = thermal[
        thermal['time_s'].between(float(row['start_s']) + settle_cutoff_s, float(row['end_s']))
        & thermal['pump_running']
    ].copy()
    if subset.empty:
        continue
    cooling = subset[subset['bulk_rate_C_s'] < 0].copy()
    power_rows.append({
        'step_id': int(step_id),
        'cmd_pct': float(row['cmd_pct']),
        'start_min': float(row['start_min']),
        'end_min': float(row['end_min']),
        'cooling_samples': int(len(cooling)),
        'median_bulk_C': float(subset['bulk_C'].median()),
        'min_bulk_C': float(subset['bulk_C'].min()),
        'median_q_inventory_w': float(cooling['q_inventory_cooling_w'].median()) if not cooling.empty else np.nan,
        'median_q_leak_w': float(subset['q_leak_model_w'].median()),
        'median_q_hx_gross_w': float(cooling['q_hx_gross_w'].median()) if not cooling.empty else np.nan,
        'median_ln_latent_power_w': float(subset['ln_latent_power_w'].median()),
        'median_ln_net_kg': float(subset['ln_net_weight_smooth_kg'].median()),
    })
power_summary = pd.DataFrame(power_rows)

display(power_summary.round(2))

fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, constrained_layout=True)

axes[0].plot(thermal['t_min'], thermal['bulk_C'], color='#111827', lw=1.25, label='TC mean bulk proxy [C]')
axes[0].plot(thermal['t_min'], thermal['temperature_c_si'], color='#64748b', lw=0.95, alpha=0.85, label='Flow-meter temp [C]')
axes[0].fill_between(thermal['t_min'], thermal['temp_min_C'], thermal['temp_max_C'], color='#f97316', alpha=0.10, label='TC min-max')
axes[0].set_ylabel('Temperature [C]')
axes[0].legend(loc='best', fontsize=8, ncols=3)
axes[0].set_title('Cooling power and warmup heat-leak history')

axes[1].plot(thermal['t_min'], thermal['q_inventory_cooling_w'], color='#2563eb', lw=1.0, alpha=0.85, label='Inventory cooling power')
axes[1].plot(thermal['t_min'], thermal['q_hx_gross_w'], color='#dc2626', lw=1.2, label='Gross HX removal estimate')
axes[1].plot(thermal['t_min'], thermal['q_leak_model_w'], color='#0f766e', lw=1.0, label='Ambient heat leak model')
axes[1].axhline(0, color='0.25', lw=0.8)
axes[1].set_ylabel('Power [W]')
axes[1].set_ylim(-250, 850)
axes[1].legend(loc='best', fontsize=8, ncols=3)

axes[2].plot(thermal['t_min'], thermal['ln_net_weight_smooth_kg'], color='#0f766e', lw=1.6, label='Smoothed net LN [kg]')
axes[2].scatter(thermal['t_min'], thermal['ln_net_weight_kg'], color='#0f766e', s=4, alpha=0.12, label='Net LN samples')
axes[2].set_ylabel('Net LN [kg]')
axes[2].legend(loc='best', fontsize=8)

axes[3].plot(thermal['t_min'], thermal['ln_latent_power_w'], color='#7c3aed', lw=1.1, label='Scale-derived LN latent power')
axes[3].plot(thermal['t_min'], thermal['pump_cmd_pct'], color='#111827', lw=0.9, alpha=0.7, label='Pump command [%]')
axes[3].fill_between(thermal['t_min'], 0, 100, where=thermal['valve'].fillna(0).ge(0.5), color='#60a5fa', alpha=0.14, step='mid', label='LN valve open')
axes[3].set_ylabel('W or percent')
axes[3].set_xlabel('Elapsed log time [min]')
axes[3].set_ylim(0, 1500)
axes[3].legend(loc='best', fontsize=8, ncols=3)

for ax in axes:
    _apply_fine_grid(ax)
    ax.set_xlim(float(thermal['t_min'].min()), float(thermal['t_min'].max()))
plt.show()

cooling_below_minus_70 = thermal[(thermal['bulk_rate_C_s'] < 0) & thermal['bulk_C'].lt(-70.0)]
warmup_pump_on = thermal[thermal['time_s'].between(float(post_ramp_40['start_s']), float(post_ramp_40['end_s']))]

display(Markdown(
    f"Below `-70 C`, median gross HX-removal estimate is "
    f"`{cooling_below_minus_70['q_hx_gross_w'].median():.0f} W` "
    f"with ambient leak contribution around `{cooling_below_minus_70['q_leak_model_w'].median():.0f} W`. "
    f"During the pump-on warmup, the model heat leak falls from about "
    f"`{preferred_warmup['fit'].heat_leak_at_temperature_C(-90.0):.0f} W` at `-90 C` "
    f"to `{preferred_warmup['fit'].heat_leak_at_temperature_C(-50.0):.0f} W` at `-50 C`."
))


,step_id,cmd_pct,start_min,end_min,cooling_samples,median_bulk_C,min_bulk_C,median_q_inventory_w,median_q_leak_w,median_q_hx_gross_w,median_ln_latent_power_w,median_ln_net_kg
0,0,20.0,20.34,24.32,114,20.04,-0.44,862.47,0.00,862.47,1500.00,35.2
1,1,30.0,24.35,48.68,597,-51.00,-68.78,259.38,75.74,340.05,830.97,28.0
2,2,40.0,48.72,57.92,190,-69.99,-75.79,216.59,96.04,309.11,637.82,23.6
3,3,30.0,57.96,62.00,66,-75.49,-79.67,241.22,101.93,341.03,558.07,22.0
4,4,40.0,62.04,101.26,797,-88.35,-95.31,122.90,115.67,237.57,482.62,18.2
5,5,45.0,101.30,104.47,81,-94.85,-95.96,272.94,122.62,396.73,683.08,14.4
6,6,40.0,104.51,167.34,945,-99.35,-102.74,71.91,127.43,198.50,387.19,10.0
7,7,45.0,167.38,167.80,0,-99.05,-99.16,NaN,127.11,NaN,386.91,3.4
8,8,50.0,167.83,168.95,0,-98.50,-98.85,NaN,126.53,NaN,235.22,3.2
9,9,55.0,168.98,169.58,0,-96.77,-97.35,NaN,124.68,NaN,92.44,3.2


Below `-70 C`, median gross HX-removal estimate is `219 W` with ambient leak contribution around `122 W`. During the pump-on warmup, the model heat leak falls from about `117 W` at `-90 C` to `75 W` at `-50 C`.

## LN-Mass Cooling Efficiency

This section treats the scale as the primary LN accounting channel. The median nonzero logged tare is applied across the full run, including early rows where the logged tare field is still zero, so `net LN = dewar gross - nonzero tare`.

The efficiency bookkeeping is split three ways:

- **Useful stored cooling**: modeled decrease in HFE plus steel internal energy between the window endpoints.
- **Estimated HX duty**: useful stored cooling plus ambient heat leak plus hydraulic pump heat during the window.
- **Available LN energy**: first the LN latent heat only, then an upper-bound latent plus sensible warming of nitrogen gas to room temperature.

That split is important because the cold run intentionally cycles the LN valve: some LN is spent fighting heat leak and maintaining temperature, not just driving the one-way cooldown.


In [8]:
cooldown_design = orca.default_cooldown_design_inputs()
LN2_DENSITY_KG_L = cooldown_design.ln2_density_kg_m3 / 1000.0
LN2_LATENT_J_KG = cooldown_design.ln2_latent_heat_j_kg
N2_GAS_CP_J_KG_K = 1040.0
LN2_SATURATION_TEMP_K = 77.0
LN2_SENSIBLE_TO_ROOM_J_KG = N2_GAS_CP_J_KG_K * (model.ambient_temp_k - LN2_SATURATION_TEMP_K)
LN2_LATENT_PLUS_ROOM_SENSIBLE_J_KG = LN2_LATENT_J_KG + LN2_SENSIBLE_TO_ROOM_J_KG

scale_quant_mask = thermal['scale_age_s'].le(2.0) & thermal['ln_net_weight_smooth_kg'].notna()
first_scale_time_s = float(thermal.loc[scale_quant_mask, 'time_s'].iloc[0])
first_scale_min = first_scale_time_s / 60.0
coldest_idx = thermal.loc[thermal['pump_running'], 'bulk_C'].idxmin()
coldest_time_s = float(thermal.loc[coldest_idx, 'time_s'])


def _thermal_energy_removed_between_temps_j(start_c, end_c, *, samples=512):
    """Positive energy means the modeled HFE+steel inventory cooled."""
    start_c = float(start_c)
    end_c = float(end_c)
    low_k = min(start_c, end_c) + 273.15
    high_k = max(start_c, end_c) + 273.15
    temp_grid_k = np.linspace(low_k, high_k, samples)
    capacity_grid = np.asarray([
        orca.thermal_capacity_j_per_k(model, float(temp_k))
        for temp_k in temp_grid_k
    ])
    magnitude_j = float(np.trapz(capacity_grid, temp_grid_k))
    return magnitude_j if start_c >= end_c else -magnitude_j


def _endpoint_subset(frame, *, start, seconds=60.0):
    if start:
        return frame[frame['time_s'].le(float(frame['time_s'].iloc[0]) + seconds)]
    return frame[frame['time_s'].ge(float(frame['time_s'].iloc[-1]) - seconds)]


def summarize_ln_energy_window(label, start_s, end_s):
    frame = thermal[thermal['time_s'].between(float(start_s), float(end_s))].copy().sort_values('time_s')
    if frame.empty:
        raise ValueError(f'No samples in window {label!r}.')

    start_frame = _endpoint_subset(frame, start=True)
    end_frame = _endpoint_subset(frame, start=False)

    start_temp_c = float(start_frame['bulk_C'].median())
    end_temp_c = float(end_frame['bulk_C'].median())
    start_ln_kg = float(start_frame['ln_net_weight_smooth_kg'].median())
    end_ln_kg = float(end_frame['ln_net_weight_smooth_kg'].median())
    ln_used_kg = start_ln_kg - end_ln_kg
    duration_s = float(frame['time_s'].iloc[-1] - frame['time_s'].iloc[0])

    useful_storage_j = _thermal_energy_removed_between_temps_j(start_temp_c, end_temp_c)
    q_leak_w = preferred_ua_w_per_k * np.clip(ambient_c - frame['bulk_C'].to_numpy(float), 0.0, None)
    ambient_leak_j = float(np.trapz(q_leak_w, frame['time_s'].to_numpy(float))) if len(frame) > 1 else np.nan
    hydraulic_power_w = (
        frame['delta_p_bar_recomputed'].to_numpy(float)
        * frame['volume_flow_lmin_si'].to_numpy(float)
        * orca_logbook.BAR_LMIN_TO_W
    )
    hydraulic_heat_j = float(np.trapz(np.clip(hydraulic_power_w, 0.0, None), frame['time_s'].to_numpy(float))) if len(frame) > 1 else np.nan
    pump_electrical_j = float(np.trapz(np.clip(frame['pump_input_power_w'].to_numpy(float), 0.0, None), frame['time_s'].to_numpy(float))) if len(frame) > 1 else np.nan

    estimated_hx_removed_j = useful_storage_j + ambient_leak_j + hydraulic_heat_j
    latent_available_j = ln_used_kg * LN2_LATENT_J_KG
    latent_plus_sensible_j = ln_used_kg * LN2_LATENT_PLUS_ROOM_SENSIBLE_J_KG

    return {
        'window': label,
        'start_min': float(start_s) / 60.0,
        'end_min': float(end_s) / 60.0,
        'duration_min': duration_s / 60.0,
        'T_start_C': start_temp_c,
        'T_end_C': end_temp_c,
        'T_min_C': float(frame['bulk_C'].min()),
        'LN_start_kg': start_ln_kg,
        'LN_end_kg': end_ln_kg,
        'LN_used_kg': ln_used_kg,
        'LN_used_L': ln_used_kg / LN2_DENSITY_KG_L,
        'avg_LN_flow_kg_min': ln_used_kg / (duration_s / 60.0) if duration_s > 0 else np.nan,
        'latent_available_MJ': latent_available_j / 1e6,
        'useful_storage_MJ': useful_storage_j / 1e6,
        'ambient_leak_MJ': ambient_leak_j / 1e6,
        'hydraulic_heat_MJ': hydraulic_heat_j / 1e6,
        'pump_electrical_MJ': pump_electrical_j / 1e6,
        'estimated_HX_removed_MJ': estimated_hx_removed_j / 1e6,
        'useful_kJ_per_kg_LN': useful_storage_j / ln_used_kg / 1000.0 if ln_used_kg > 0 else np.nan,
        'HX_kJ_per_kg_LN': estimated_hx_removed_j / ln_used_kg / 1000.0 if ln_used_kg > 0 else np.nan,
        'avg_latent_available_W': latent_available_j / duration_s if duration_s > 0 else np.nan,
        'avg_HX_removed_W': estimated_hx_removed_j / duration_s if duration_s > 0 else np.nan,
        'useful_over_latent_pct': 100.0 * useful_storage_j / latent_available_j if latent_available_j > 0 else np.nan,
        'HX_over_latent_pct': 100.0 * estimated_hx_removed_j / latent_available_j if latent_available_j > 0 else np.nan,
        'HX_over_latent_plus_room_sensible_pct': 100.0 * estimated_hx_removed_j / latent_plus_sensible_j if latent_plus_sensible_j > 0 else np.nan,
        'hydraulic_over_HX_pct': 100.0 * hydraulic_heat_j / estimated_hx_removed_j if estimated_hx_removed_j > 0 else np.nan,
    }

below_minus_70 = thermal[
    thermal['pump_running']
    & thermal['bulk_C'].lt(-70.0)
    & thermal['time_s'].ge(first_scale_time_s)
]

energy_windows = [
    summarize_ln_energy_window('full-run scale to coldest TC mean', first_scale_time_s, coldest_time_s),
    summarize_ln_energy_window('30% command to coldest TC mean', float(step_windows.loc[1, 'start_s']), coldest_time_s),
    summarize_ln_energy_window('below -70 C to coldest TC mean', float(below_minus_70['time_s'].iloc[0]), coldest_time_s),
    summarize_ln_energy_window('cold 40% hold plus 45-60% ramp', float(step_windows.loc[6, 'start_s']), float(step_windows.loc[10, 'end_s'])),
    summarize_ln_energy_window('full-run scale to pump off', first_scale_time_s, float(thermal.loc[thermal['pump_running'], 'time_s'].max())),
    summarize_ln_energy_window('full-run scale full log', first_scale_time_s, float(thermal['time_s'].iloc[-1])),
]
ln_efficiency_summary = pd.DataFrame(energy_windows).set_index('window')

main_cols = [
    'duration_min',
    'T_start_C',
    'T_end_C',
    'T_min_C',
    'LN_used_kg',
    'LN_used_L',
    'avg_LN_flow_kg_min',
    'latent_available_MJ',
    'useful_storage_MJ',
    'ambient_leak_MJ',
    'hydraulic_heat_MJ',
    'estimated_HX_removed_MJ',
    'avg_latent_available_W',
    'avg_HX_removed_W',
    'useful_over_latent_pct',
    'HX_over_latent_pct',
    'HX_over_latent_plus_room_sensible_pct',
]
display(ln_efficiency_summary[main_cols].round(3))

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.8), constrained_layout=True)
main_window = ln_efficiency_summary.loc['full-run scale to coldest TC mean']
plot_frame = thermal[thermal['time_s'].between(first_scale_time_s, coldest_time_s)].copy()

axes[0, 0].plot(plot_frame['t_min'], plot_frame['ln_net_weight_smooth_kg'], color='#0f766e', lw=2.0)
axes[0, 0].scatter(plot_frame['t_min'], plot_frame['ln_net_weight_kg'], s=5, color='#0f766e', alpha=0.14)
axes[0, 0].set_ylabel('Net LN in dewar [kg]')
axes[0, 0].set_title('LN use from dewar minus tare')

axes[0, 1].plot(plot_frame['t_min'], plot_frame['bulk_C'], color='#111827', lw=1.4, label='Bulk proxy')
axes[0, 1].set_ylabel('Bulk temperature [C]')
axes[0, 1].set_title('Cooling delivered to HFE plus hardware')
axes_temp2 = axes[0, 1].twinx()
axes_temp2.plot(plot_frame['t_min'], plot_frame['ln_net_weight_smooth_kg'], color='#0f766e', lw=1.4, alpha=0.75, label='Net LN')
axes_temp2.set_ylabel('Net LN [kg]')

bar_labels = ['Useful\nstored', 'Heat\nleak', 'Hydraulic\nheat', 'Latent\navailable']
bar_values = [
    main_window['useful_storage_MJ'],
    main_window['ambient_leak_MJ'],
    main_window['hydraulic_heat_MJ'],
    main_window['latent_available_MJ'],
]
bar_colors = ['#2563eb', '#0f766e', '#7c3aed', '#64748b']
axes[1, 0].bar(bar_labels, bar_values, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[1, 0].set_ylabel('Energy [MJ]')
axes[1, 0].set_title('Main cooldown energy budget')

short = ln_efficiency_summary.loc[
    ['full-run scale to coldest TC mean', 'below -70 C to coldest TC mean', 'cold 40% hold plus 45-60% ramp'],
    ['useful_over_latent_pct', 'HX_over_latent_pct', 'HX_over_latent_plus_room_sensible_pct'],
].rename(index={
    'full-run scale to coldest TC mean': 'Full cooldown',
    'below -70 C to coldest TC mean': 'Below -70 C',
    'cold 40% hold plus 45-60% ramp': 'Cold hold+ramp',
})
short.plot(kind='bar', ax=axes[1, 1], color=['#2563eb', '#dc2626', '#94a3b8'], edgecolor='black', linewidth=0.6)
axes[1, 1].set_ylabel('Efficiency [%]')
axes[1, 1].set_title('LN utilization metrics')
axes[1, 1].legend(['Useful / latent', 'HX / latent', 'HX / latent+sensible'], fontsize=8, loc='best')
axes[1, 1].tick_params(axis='x', rotation=20)

for ax in [axes[0, 0], axes[0, 1], axes_temp2, axes[1, 0], axes[1, 1]]:
    _apply_fine_grid(ax)
axes[0, 0].set_xlabel('Elapsed log time [min]')
axes[0, 1].set_xlabel('Elapsed log time [min]')
plt.show()

main = ln_efficiency_summary.loc['full-run scale to coldest TC mean']
display(Markdown(
    f"Using the median nonzero tare across the full log, from the first valid scale point (`{first_scale_min:.2f} min`) to the coldest TC mean "
    f"(`{coldest_time_s / 60.0:.2f} min`), the dewar lost `{main['LN_used_kg']:.1f} kg` "
    f"(`{main['LN_used_L']:.1f} L`) of LN2. Latent-only energy available was "
    f"`{main['latent_available_MJ']:.2f} MJ`. The modeled useful stored cooling was "
    f"`{main['useful_storage_MJ']:.2f} MJ` (`{main['useful_over_latent_pct']:.1f}%` of latent), "
    f"while the estimated HX duty including heat leak and hydraulic heat was "
    f"`{main['estimated_HX_removed_MJ']:.2f} MJ` (`{main['HX_over_latent_pct']:.1f}%` of latent). "
    f"If nitrogen gas sensible warming to room temperature is counted as an upper-bound available energy, "
    f"that HX utilization becomes `{main['HX_over_latent_plus_room_sensible_pct']:.1f}%`."
))


,duration_min,T_start_C,T_end_C,T_min_C,LN_used_kg,LN_used_L,avg_LN_flow_kg_min,latent_available_MJ,useful_storage_MJ,ambient_leak_MJ,hydraulic_heat_MJ,estimated_HX_removed_MJ,avg_latent_available_W,avg_HX_removed_W,useful_over_latent_pct,HX_over_latent_pct,HX_over_latent_plus_room_sensible_pct
window,,,,,,,,,,,,,,,,,
full-run scale to coldest TC mean,122.977,21.401,-102.218,-102.737,29.0,35.891,0.236,4.785,1.138,0.768,0.021,1.927,648.494,261.144,23.792,40.269,17.053
30% command to coldest TC mean,118.968,-8.645,-102.218,-102.737,24.6,30.446,0.207,4.059,0.853,0.766,0.021,1.640,568.640,229.781,21.015,40.409,17.112
below -70 C to coldest TC mean,91.915,-70.839,-102.218,-102.737,14.6,18.069,0.159,2.409,0.279,0.647,0.020,0.945,436.819,171.428,11.571,39.245,16.619
cold 40% hold plus 45-60% ramp,68.240,-95.087,-91.339,-102.737,11.0,13.614,0.161,1.815,-0.033,0.518,0.021,0.506,443.291,123.643,-1.823,27.892,11.811
full-run scale to pump off,273.113,21.401,-27.865,-102.737,35.0,43.317,0.128,5.775,0.466,1.559,0.049,2.074,352.418,126.560,8.062,35.912,15.208
full-run scale full log,330.042,21.401,-7.711,-102.737,35.0,43.317,0.106,5.775,0.277,1.688,0.049,2.013,291.629,101.660,4.791,34.859,14.762


Using the median nonzero tare across the full log, from the first valid scale point (`20.34 min`) to the coldest TC mean (`143.32 min`), the dewar lost `29.0 kg` (`35.9 L`) of LN2. Latent-only energy available was `4.79 MJ`. The modeled useful stored cooling was `1.14 MJ` (`23.8%` of latent), while the estimated HX duty including heat leak and hydraulic heat was `1.93 MJ` (`40.3%` of latent). If nitrogen gas sensible warming to room temperature is counted as an upper-bound available energy, that HX utilization becomes `17.1%`.

## Viscosity-Proxy Search Below -70 C

This section ranks logged signals against the HFE-7200 viscosity reference values used in `HFE_properties.ipynb` (`analysis/docs/HFE Freezing Data.pdf`, 3M/Novec 7200 table). The cleanest physics proxy is still temperature or density, but the pump-side candidates are useful because they are independent telemetry channels. To avoid command-speed effects, the ranking below uses only 40 percent pump-command samples colder than `-70 C`.

The log has one pump-inlet pressure channel, so "inlet pressure spread" here means variation of that channel at fixed command: a 60 s rolling standard deviation and a robust 10-90 percent spread inside 5 C temperature bands.

In [9]:
reference_viscosity = orca_logbook.REFERENCE_3M_N7200.sort_values('temperature_c').copy()


def hfe7200_density_kg_m3_3m(temperature_c):
    return 1000.0 * (
        orca_logbook.HFE_7200_DENSITY_INTERCEPT_G_ML
        - orca_logbook.HFE_7200_DENSITY_SLOPE_G_ML_PER_C * np.asarray(temperature_c, dtype=float)
    )


reference_viscosity['density_kg_m3_3m'] = hfe7200_density_kg_m3_3m(reference_viscosity['temperature_c'])
reference_viscosity['dynamic_viscosity_cP'] = (
    reference_viscosity['kinematic_viscosity_cSt']
    * reference_viscosity['density_kg_m3_3m']
    / 1000.0
)
viscosity_interp = PchipInterpolator(
    reference_viscosity['temperature_c'].to_numpy(float),
    np.log(reference_viscosity['kinematic_viscosity_cSt'].to_numpy(float)),
    extrapolate=True,
)


def hfe7200_kinematic_viscosity_cst(temperature_c):
    return np.exp(viscosity_interp(np.asarray(temperature_c, dtype=float)))


visc = thermal.copy().sort_values('time_s').reset_index(drop=True)
visc['nu_hfe7200_ref_cSt'] = hfe7200_kinematic_viscosity_cst(visc['bulk_C'])
visc['mu_hfe7200_ref_cP'] = visc['nu_hfe7200_ref_cSt'] * visc['density_kg_m3_si'] / 1000.0
visc['delivered_ml_per_rev'] = visc['volume_flow_lmin_si'] * 1000.0 / (60.0 * visc['pump_freq_hz'].replace(0.0, np.nan))
visc['flow_per_hz_lmin_hz'] = visc['volume_flow_lmin_si'] / visc['pump_freq_hz'].replace(0.0, np.nan)
visc['hydraulic_power_w'] = visc['delta_p_bar_recomputed'] * visc['volume_flow_lmin_si'] * orca_logbook.BAR_LMIN_TO_W
visc['specific_input_energy_j_l'] = 60.0 * visc['pump_input_power_w'] / visc['volume_flow_lmin_si'].replace(0.0, np.nan)
visc['power_per_lmin_w_lmin'] = visc['pump_input_power_w'] / visc['volume_flow_lmin_si'].replace(0.0, np.nan)
visc['current_per_lmin_a_lmin'] = visc['pump_output_current_a'] / visc['volume_flow_lmin_si'].replace(0.0, np.nan)

VISCOSITY_PROXY_PUMP_CMD_PCT = 40.0
VISCOSITY_PROXY_PUMP_CMD_TOL_PCT = 0.25
viscosity_40pct_mask = visc['pump_cmd_pct'].sub(VISCOSITY_PROXY_PUMP_CMD_PCT).abs().le(VISCOSITY_PROXY_PUMP_CMD_TOL_PCT)

proxy_subset = visc[
    visc['pump_running']
    & viscosity_40pct_mask
    & visc['time_s'].ge(BYPASS_CLOSED_TIME_S)
    & visc['bulk_C'].lt(-70.0)
].copy().sort_values('time_s')

pressure_time_index = pd.to_timedelta(proxy_subset['time_s'].to_numpy(float), unit='s')
pressure_rolling_std = (
    pd.Series(proxy_subset['pump_pressure_before_bar_abs'].to_numpy(float), index=pressure_time_index)
    .rolling('60s', center=True, min_periods=10)
    .std()
)
proxy_subset['pump_inlet_pressure_std_60s_bar'] = pressure_rolling_std.to_numpy()

warm_ref_mask = proxy_subset['bulk_C'].between(-75.0, -70.0)
cold_ref_mask = proxy_subset['bulk_C'].lt(-90.0)
warm_ref_nu = float(proxy_subset.loc[warm_ref_mask, 'nu_hfe7200_ref_cSt'].median())
cold_ref_nu = float(proxy_subset.loc[cold_ref_mask, 'nu_hfe7200_ref_cSt'].median())
warm_ref_mu = float(proxy_subset.loc[warm_ref_mask, 'mu_hfe7200_ref_cP'].median())
cold_ref_mu = float(proxy_subset.loc[cold_ref_mask, 'mu_hfe7200_ref_cP'].median())

proxy_signals = {
    'density_kg_m3_si': 'Density',
    'mass_flow_kgmin_si': 'Mass flow',
    'volume_flow_lmin_si': 'Volume flow',
    'delivered_ml_per_rev': 'Delivered mL/rev',
    'delta_p_bar_recomputed': 'Pump pressure rise',
    'hydraulic_power_w': 'Hydraulic power',
    'pump_pressure_before_bar_abs': 'Pump inlet pressure',
    'pump_inlet_pressure_std_60s_bar': 'Pump inlet pressure 60 s spread',
    'pump_input_power_w': 'Pump input power',
    'specific_input_energy_j_l': 'Specific input energy',
    'current_per_lmin_a_lmin': 'Current per L/min',
}

proxy_rows = []
for column, label in proxy_signals.items():
    x = pd.to_numeric(proxy_subset[column], errors='coerce')
    y = proxy_subset['nu_hfe7200_ref_cSt']
    valid = x.notna() & y.notna() & np.isfinite(x.to_numpy(float)) & np.isfinite(y.to_numpy(float))
    if int(valid.sum()) < 10:
        continue
    warm = float(x[warm_ref_mask].median())
    cold = float(x[cold_ref_mask].median())
    proxy_rows.append({
        'signal': column,
        'label': label,
        'samples': int(valid.sum()),
        'pearson_vs_3m_nu': float(x[valid].corr(y[valid], method='pearson')),
        'spearman_vs_3m_nu': float(x[valid].corr(y[valid], method='spearman')),
        'median_at_minus_70_to_75': warm,
        'median_below_minus_90': cold,
        'pct_change_cold_vs_warm': 100.0 * (cold - warm) / warm if np.isfinite(warm) and abs(warm) > 1e-12 else np.nan,
        '3m_nu_minus_70_to_75_cSt': warm_ref_nu,
        '3m_nu_below_minus_90_cSt': cold_ref_nu,
        '3m_nu_pct_change': 100.0 * (cold_ref_nu - warm_ref_nu) / warm_ref_nu,
        '3m_mu_minus_70_to_75_cP': warm_ref_mu,
        '3m_mu_below_minus_90_cP': cold_ref_mu,
    })
proxy_summary = (
    pd.DataFrame(proxy_rows)
    .sort_values('spearman_vs_3m_nu', key=lambda series: series.abs(), ascending=False)
    .set_index('signal')
)

display(reference_viscosity[['temperature_c', 'kinematic_viscosity_cSt', 'dynamic_viscosity_cP']].round(3))
display(proxy_summary.round(4))

banded_proxy = proxy_subset.copy()
banded_proxy['bulk_temp_band_C'] = pd.cut(
    banded_proxy['bulk_C'],
    bins=[-105.0, -100.0, -95.0, -90.0, -85.0, -80.0, -75.0, -70.0],
    include_lowest=True,
)
band_summary = banded_proxy.groupby('bulk_temp_band_C', observed=True).agg(
    samples=('time_s', 'size'),
    median_bulk_C=('bulk_C', 'median'),
    ref_nu_cSt=('nu_hfe7200_ref_cSt', 'median'),
    ref_mu_cP=('mu_hfe7200_ref_cP', 'median'),
    density_kg_m3=('density_kg_m3_si', 'median'),
    volume_flow_lmin=('volume_flow_lmin_si', 'median'),
    delta_p_bar=('delta_p_bar_recomputed', 'median'),
    hydraulic_power_w=('hydraulic_power_w', 'median'),
    pump_inlet_bar_abs=('pump_pressure_before_bar_abs', 'median'),
    pump_inlet_p10_bar_abs=('pump_pressure_before_bar_abs', lambda s: s.quantile(0.10)),
    pump_inlet_p90_bar_abs=('pump_pressure_before_bar_abs', lambda s: s.quantile(0.90)),
    pump_inlet_std_bar=('pump_pressure_before_bar_abs', 'std'),
    pump_inlet_rolling_std_60s_bar=('pump_inlet_pressure_std_60s_bar', 'median'),
)
band_summary['pump_inlet_p90_p10_spread_mbar'] = 1000.0 * (
    band_summary['pump_inlet_p90_bar_abs'] - band_summary['pump_inlet_p10_bar_abs']
)
band_summary['pump_inlet_std_mbar'] = 1000.0 * band_summary['pump_inlet_std_bar']
band_summary['pump_inlet_rolling_std_60s_mbar'] = 1000.0 * band_summary['pump_inlet_rolling_std_60s_bar']
display(band_summary.round(3))


def _loglog_fit_summary(frame, x_col, y_col, label):
    valid = (
        frame[x_col].notna()
        & frame[y_col].notna()
        & np.isfinite(frame[x_col].to_numpy(float))
        & np.isfinite(frame[y_col].to_numpy(float))
        & frame[x_col].gt(0.0)
        & frame[y_col].gt(0.0)
        & frame['samples'].ge(20)
    )
    fit_frame = frame.loc[valid, [x_col, y_col]].copy()
    if len(fit_frame) < 3:
        return {
            'metric': label,
            'bands': int(len(fit_frame)),
            'loglog_exponent_vs_3m_nu': np.nan,
            'r_squared_loglog': np.nan,
            'spearman_vs_3m_nu': np.nan,
        }
    log_x = np.log(fit_frame[x_col].to_numpy(float))
    log_y = np.log(fit_frame[y_col].to_numpy(float))
    slope, intercept = np.polyfit(log_x, log_y, 1)
    pred = intercept + slope * log_x
    ss_res = float(np.sum((log_y - pred) ** 2))
    ss_tot = float(np.sum((log_y - np.mean(log_y)) ** 2))
    return {
        'metric': label,
        'bands': int(len(fit_frame)),
        'loglog_exponent_vs_3m_nu': float(slope),
        'r_squared_loglog': 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        'spearman_vs_3m_nu': float(fit_frame[x_col].corr(fit_frame[y_col], method='spearman')),
    }


pressure_spread_fit = pd.DataFrame([
    _loglog_fit_summary(band_summary, 'ref_nu_cSt', 'pump_inlet_p90_p10_spread_mbar', 'inlet 10-90% band spread'),
    _loglog_fit_summary(band_summary, 'ref_nu_cSt', 'pump_inlet_std_mbar', 'inlet band standard deviation'),
    _loglog_fit_summary(band_summary, 'ref_nu_cSt', 'pump_inlet_rolling_std_60s_mbar', 'median 60 s rolling std'),
]).set_index('metric')
display(pressure_spread_fit.round(3))

fig, axes = plt.subplots(3, 2, figsize=(13.8, 12.0), constrained_layout=True)
plot_specs = [
    ('density_kg_m3_si', 'Density [kg/m3]', '#2563eb'),
    ('delta_p_bar_recomputed', 'Pump pressure rise [bar]', '#dc2626'),
    ('hydraulic_power_w', 'Hydraulic power [W]', '#7c3aed'),
    ('pump_pressure_before_bar_abs', 'Pump inlet pressure [bar abs]', '#0f766e'),
    ('pump_inlet_pressure_std_60s_bar', 'Inlet pressure 60 s std [bar]', '#0891b2'),
    ('specific_input_energy_j_l', 'Specific input energy [J/L]', '#ea580c'),
]
reference_temp_grid = np.linspace(-105.0, -70.0, 220)
reference_points_cold = reference_viscosity[reference_viscosity['temperature_c'].between(-105.0, -70.0)].copy()

for ax, (column, ylabel, color) in zip(axes.ravel(), plot_specs):
    ax.scatter(proxy_subset['bulk_C'], proxy_subset[column], s=8, alpha=0.26, color=color)
    grouped = proxy_subset.groupby(pd.cut(proxy_subset['bulk_C'], np.arange(-105, -68, 5)), observed=True)
    mids = []
    medians = []
    for interval, group in grouped:
        if group.empty:
            continue
        mids.append(0.5 * (interval.left + interval.right))
        medians.append(group[column].median())
    ax.plot(mids, medians, color='black', lw=2.0, marker='o', ms=4, label='5 C median')
    ax_ref = ax.twinx()
    ax_ref.plot(
        reference_temp_grid,
        hfe7200_kinematic_viscosity_cst(reference_temp_grid),
        color='0.35',
        lw=1.4,
        ls=':',
        label='3M HFE-7200 nu',
    )
    ax_ref.scatter(
        reference_points_cold['temperature_c'],
        reference_points_cold['kinematic_viscosity_cSt'],
        s=32,
        color='0.35',
        marker='s',
        zorder=4,
        label='3M points',
    )
    ax_ref.set_yscale('log')
    ax_ref.set_ylabel('3M ref nu [cSt]', color='0.35')
    ax_ref.tick_params(axis='y', colors='0.35')
    ax.set_xlabel('TC mean bulk temperature [C]')
    ax.set_ylabel(ylabel)
    ax.set_title(proxy_signals[column])
    ax.invert_xaxis()
    lines = ax.get_lines() + ax_ref.get_lines()
    labels = [line.get_label() for line in lines]
    ax.legend(lines, labels, loc='best', fontsize=8)
    _apply_fine_grid(ax)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), constrained_layout=True)
spread_plot = band_summary.reset_index().dropna(subset=['ref_nu_cSt', 'pump_inlet_p90_p10_spread_mbar'])
axes[0].plot(
    spread_plot['ref_nu_cSt'],
    spread_plot['pump_inlet_p90_p10_spread_mbar'],
    marker='o',
    color='#0891b2',
    lw=2.0,
    label='10-90% band spread',
)
axes[0].plot(
    spread_plot['ref_nu_cSt'],
    spread_plot['pump_inlet_rolling_std_60s_mbar'],
    marker='s',
    color='#7c3aed',
    lw=2.0,
    label='Median 60 s rolling std',
)
for _, row in spread_plot.iterrows():
    axes[0].annotate(f"{row['median_bulk_C']:.0f} C", (row['ref_nu_cSt'], row['pump_inlet_p90_p10_spread_mbar']), xytext=(4, 5), textcoords='offset points', fontsize=8)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('3M HFE-7200 reference kinematic viscosity [cSt]')
axes[0].set_ylabel('Pump inlet pressure spread [mbar]')
axes[0].set_title('Does inlet pressure spread scale with expected viscosity?')
axes[0].legend(loc='best', fontsize=8)

axes[1].fill_between(
    band_summary['median_bulk_C'],
    band_summary['pump_inlet_p10_bar_abs'],
    band_summary['pump_inlet_p90_bar_abs'],
    color='#0891b2',
    alpha=0.20,
    label='Inlet p10-p90',
)
axes[1].plot(band_summary['median_bulk_C'], band_summary['pump_inlet_bar_abs'], marker='o', color='#0f766e', lw=2.0, label='Inlet median')
axes_1_ref = axes[1].twinx()
axes_1_ref.plot(reference_temp_grid, hfe7200_kinematic_viscosity_cst(reference_temp_grid), color='0.35', ls=':', lw=1.6, label='3M nu')
axes_1_ref.scatter(reference_points_cold['temperature_c'], reference_points_cold['kinematic_viscosity_cSt'], s=35, color='0.35', marker='s', zorder=4)
axes_1_ref.set_yscale('log')
axes_1_ref.set_ylabel('3M ref nu [cSt]', color='0.35')
axes_1_ref.tick_params(axis='y', colors='0.35')
axes[1].invert_xaxis()
axes[1].set_xlabel('TC mean bulk temperature [C]')
axes[1].set_ylabel('Pump inlet pressure [bar abs]')
axes[1].set_title('Inlet pressure median and spread during 40% cold hold')
lines = axes[1].get_lines() + axes_1_ref.get_lines()
axes[1].legend(lines, [line.get_label() for line in lines], loc='best', fontsize=8)

for ax in [axes[0], axes[1], axes_1_ref]:
    _apply_fine_grid(ax)
plt.show()

spread_main = pressure_spread_fit.loc['inlet 10-90% band spread']
rolling_main = pressure_spread_fit.loc['median 60 s rolling std']
display(Markdown(
    f"At fixed 40 percent command below `-70 C`, the pump-inlet pressure spread does increase with the "
    f"3M HFE-7200 viscosity reference. The 5 C banded 10-90 percent inlet spread gives a log-log "
    f"exponent of `{spread_main['loglog_exponent_vs_3m_nu']:.2f}` versus reference viscosity with "
    f"`R^2 = {spread_main['r_squared_loglog']:.2f}` and Spearman `{spread_main['spearman_vs_3m_nu']:.2f}`. "
    f"The median 60 s rolling inlet-pressure standard deviation gives exponent "
    f"`{rolling_main['loglog_exponent_vs_3m_nu']:.2f}`, `R^2 = {rolling_main['r_squared_loglog']:.2f}`. "
    f"So the inlet-pressure spread is directionally viscosity-like, but it is a secondary proxy: it also carries "
    f"instrument quantization, suction-side hydraulics, and any pressure-control/transient behavior."
))


,temperature_c,kinematic_viscosity_cSt,dynamic_viscosity_cP
10,-120.0,64.47,113.300
9,-100.0,12.47,21.341
8,-70.0,3.72,6.109
7,-60.0,2.48,4.016
6,-50.0,1.84,2.937
5,-40.0,1.42,2.234
4,-30.0,1.14,1.767
3,-20.0,0.93,1.420
2,-10.0,0.78,1.173
1,0.0,0.67,0.992


,label,samples,pearson_vs_3m_nu,spearman_vs_3m_nu,median_at_minus_70_to_75,median_below_minus_90,pct_change_cold_vs_warm,3m_nu_minus_70_to_75_cSt,3m_nu_below_minus_90_cSt,3m_nu_pct_change,3m_mu_minus_70_to_75_cP,3m_mu_below_minus_90_cP
signal,,,,,,,,,,,,
density_kg_m3_si,Density,3620,0.9305,0.9337,1622.0000,1663.0000,2.5277,4.1177,11.2588,173.4234,6.677,18.7017
mass_flow_kgmin_si,Mass flow,3620,0.8038,0.9219,4.3559,4.5422,4.2766,4.1177,11.2588,173.4234,6.677,18.7017
delivered_ml_per_rev,Delivered mL/rev,3620,0.8595,0.8586,1.5908,1.6188,1.7634,4.1177,11.2588,173.4234,6.677,18.7017
volume_flow_lmin_si,Volume flow,3620,0.5338,0.8574,2.6857,2.7322,1.7303,4.1177,11.2588,173.4234,6.677,18.7017
current_per_lmin_a_lmin,Current per L/min,3620,-0.5902,-0.8260,0.9507,0.9345,-1.7091,4.1177,11.2588,173.4234,6.677,18.7017
hydraulic_power_w,Hydraulic power,3620,0.7258,0.7291,2.7520,4.5252,64.4319,4.1177,11.2588,173.4234,6.677,18.7017
delta_p_bar_recomputed,Pump pressure rise,3620,0.7171,0.7217,0.6150,0.9960,61.9512,4.1177,11.2588,173.4234,6.677,18.7017
pump_pressure_before_bar_abs,Pump inlet pressure,3620,-0.5143,-0.5193,2.2120,1.9690,-10.9855,4.1177,11.2588,173.4234,6.677,18.7017
pump_input_power_w,Pump input power,3620,0.4390,0.4695,69.0000,70.0000,1.4493,4.1177,11.2588,173.4234,6.677,18.7017


,samples,median_bulk_C,ref_nu_cSt,ref_mu_cP,density_kg_m3,volume_flow_lmin,delta_p_bar,hydraulic_power_w,pump_inlet_bar_abs,pump_inlet_p10_bar_abs,pump_inlet_p90_bar_abs,pump_inlet_std_bar,pump_inlet_rolling_std_60s_bar,pump_inlet_p90_p10_spread_mbar,pump_inlet_std_mbar,pump_inlet_rolling_std_60s_mbar
bulk_temp_band_C,,,,,,,,,,,,,,,,
"(-105.001, -100.0]",779,-101.459,13.571,22.660,1669.0,2.742,1.134,5.196,1.920,1.618,2.127,0.192,0.172,508.8,192.218,171.958
"(-100.0, -95.0]",874,-97.661,11.014,18.273,1661.0,2.733,0.988,4.506,1.959,1.657,2.300,0.228,0.224,643.0,228.088,224.337
"(-95.0, -90.0]",601,-92.607,8.691,14.326,1646.0,2.716,0.802,3.642,2.066,1.871,2.320,0.174,0.167,449.0,173.754,166.860
"(-90.0, -85.0]",454,-87.721,7.129,11.678,1640.0,2.710,0.752,3.381,2.144,1.910,2.346,0.172,0.128,436.0,172.012,127.680
"(-85.0, -80.0]",369,-82.655,5.919,9.683,1636.0,2.702,0.683,3.053,2.164,1.959,2.349,0.157,0.118,390.0,157.437,117.861
"(-80.0, -75.0]",276,-77.486,4.928,8.031,1631.0,2.696,0.655,2.934,2.208,1.983,2.368,0.148,0.154,385.0,148.164,153.948
"(-75.0, -70.0]",267,-72.588,4.118,6.677,1622.0,2.686,0.615,2.752,2.212,2.056,2.417,0.134,0.135,361.0,133.962,135.159


,bands,loglog_exponent_vs_3m_nu,r_squared_loglog,spearman_vs_3m_nu
metric,,,,
inlet 10-90% band spread,7,0.397,0.741,0.964
inlet band standard deviation,7,0.367,0.812,0.964
median 60 s rolling std,7,0.335,0.440,0.679


At fixed 40 percent command below `-70 C`, the pump-inlet pressure spread does increase with the 3M HFE-7200 viscosity reference. The 5 C banded 10-90 percent inlet spread gives a log-log exponent of `0.40` versus reference viscosity with `R^2 = 0.74` and Spearman `0.96`. The median 60 s rolling inlet-pressure standard deviation gives exponent `0.34`, `R^2 = 0.44`. So the inlet-pressure spread is directionally viscosity-like, but it is a secondary proxy: it also carries instrument quantization, suction-side hydraulics, and any pressure-control/transient behavior.

## Room-Anchor Defensible Viscosity Proxies

This section uses only the room-temperature HFE-7200 viscosity value from 3M (`nu_25 = 0.41 cSt`) plus the measured room-temperature `40%` pump-speed anchor from the `2026-04-22 14:33:45` recirculation log. The table separates proxies that follow a fixed-geometry hydraulic scaling from signals that are useful diagnostics but should not be treated as viscosity measurements without an additional pump/efficiency/noise model.

In [10]:

ROOM_REFERENCE_LOG_PATH = REPO_ROOT / 'data' / 'raw' / 'recirculation' / 'log_20260422_143345.csv'
ROOM_REFERENCE_PUMP_CMD_PCT = 40.0
ROOM_REFERENCE_PUMP_CMD_TOL_PCT = 0.25
ROOM_REFERENCE_TEMP_MIN_C = 20.0
ROOM_REFERENCE_TEMP_MAX_C = 26.0
ROOM_REFERENCE_PIN_SPREAD_WINDOW_S = 120.0
PHYSICAL_PROXY_SMOOTHING_WINDOW_S = 120.0
VISCOSITY_PROXY_TEMP_MIN_C = -120.0
VISCOSITY_PROXY_TEMP_MAX_C = 0.0
VISCOSITY_PROXY_BIN_WIDTH_C = 2.5
HFE7200_ROOM_TEMP_C = 25.0
HFE7200_ROOM_KINEMATIC_VISCOSITY_CST = float(
    reference_viscosity.loc[
        reference_viscosity['temperature_c'].eq(HFE7200_ROOM_TEMP_C),
        'kinematic_viscosity_cSt',
    ].iloc[0]
)


def _add_bulk_temperature(frame, valid_temp_cols):
    frame = frame.copy()
    if valid_temp_cols:
        frame['bulk_C'] = frame[list(valid_temp_cols)].median(axis=1)
    elif 'temperature_c_si' in frame.columns:
        frame['bulk_C'] = frame['temperature_c_si']
    else:
        frame['bulk_C'] = np.nan
    return frame


def _add_hydraulic_proxy_primitives(frame):
    frame = frame.copy()
    q_lmin = frame['volume_flow_lmin_si'].replace(0.0, np.nan)
    frame['hydraulic_resistance_bar_per_lmin'] = frame['delta_p_bar_recomputed'] / q_lmin
    frame['hydraulic_power_w'] = frame['delta_p_bar_recomputed'] * q_lmin * orca_logbook.BAR_LMIN_TO_W
    frame['hydraulic_power_per_flow2_w_per_lmin2'] = frame['hydraulic_power_w'] / (q_lmin ** 2)
    frame['specific_input_energy_j_l'] = 60.0 * frame['pump_input_power_w'] / q_lmin
    return frame


def _rolling_time_stat(frame, column, window_s, *, agg='median', min_periods=10):
    series = pd.Series(
        pd.to_numeric(frame[column], errors='coerce').to_numpy(float),
        index=pd.to_timedelta(frame['time_s'].to_numpy(float), unit='s'),
    )
    rolled = series.rolling(f'{float(window_s):.0f}s', center=True, min_periods=min_periods)
    if agg == 'median':
        return rolled.median().to_numpy()
    if agg == 'mean':
        return rolled.mean().to_numpy()
    if agg == 'std':
        return rolled.std().to_numpy()
    raise ValueError(f'Unsupported rolling aggregation: {agg}')


room_reference_review = orca.prepare_flow_log_review(ROOM_REFERENCE_LOG_PATH, density_bounds=DENSITY_BOUNDS_KG_M3, tc_calibration_path=TC_CALIBRATION_PATH)
room_reference = _add_bulk_temperature(room_reference_review.data.sort_values('time_s').reset_index(drop=True), room_reference_review.valid_temp_cols)
room_reference = _add_hydraulic_proxy_primitives(room_reference)
room_reference['pump_inlet_pressure_spread_120s_bar'] = _rolling_time_stat(
    room_reference,
    'pump_pressure_before_bar_abs',
    ROOM_REFERENCE_PIN_SPREAD_WINDOW_S,
    agg='std',
    min_periods=10,
)
room_reference_mask = (
    room_reference['pump_running']
    & room_reference['pump_cmd_pct'].sub(ROOM_REFERENCE_PUMP_CMD_PCT).abs().le(ROOM_REFERENCE_PUMP_CMD_TOL_PCT)
    & room_reference['temperature_c_si'].between(ROOM_REFERENCE_TEMP_MIN_C, ROOM_REFERENCE_TEMP_MAX_C)
    & room_reference['volume_flow_lmin_si'].gt(0.0)
    & room_reference['delta_p_bar_recomputed'].gt(0.0)
)
room_reference_anchor = room_reference.loc[room_reference_mask].median(numeric_only=True)
room_reference_samples = int(room_reference_mask.sum())
room_reference_time_min = room_reference.loc[room_reference_mask, 'time_s'].agg(['min', 'max']) / 60.0

rho_ref = float(room_reference_anchor['density_kg_m3_si'])
q_ref = float(room_reference_anchor['volume_flow_lmin_si'])
dp_ref = float(room_reference_anchor['delta_p_bar_recomputed'])
rhyd_ref = float(room_reference_anchor['hydraulic_resistance_bar_per_lmin'])
phyd_ref = float(room_reference_anchor['hydraulic_power_w'])
phyd_q2_ref = float(room_reference_anchor['hydraulic_power_per_flow2_w_per_lmin2'])
ein_ref = float(room_reference_anchor['specific_input_energy_j_l'])
pin_spread_ref = float(room_reference_anchor['pump_inlet_pressure_spread_120s_bar'])
room_temp_ref = float(room_reference_anchor['temperature_c_si'])
bulk_temp_ref = float(room_reference_anchor['bulk_C'])
pump_freq_ref = float(room_reference_anchor['pump_freq_hz'])

proxy_defensibility_table = pd.DataFrame([
    {
        'proxy': 'Density',
        '0422 room 40% anchor': f'{rho_ref:.2f} kg/m3',
        'scaling shown': 'index only: nu_25 * rho/rho_ref',
        'defensible as viscosity?': 'No',
        'scientific basis / limitation': 'Density is a state/temperature proxy. It is needed to convert dynamic to kinematic viscosity, but density alone has no general viscosity law.',
    },
    {
        'proxy': 'Pump pressure rise',
        '0422 room 40% anchor': f'DeltaP={dp_ref:.3f} bar; DeltaP/Q={rhyd_ref:.6f} bar/(L/min)',
        'scaling shown': 'nu_hat = nu_25 * (rho_ref/rho) * [(DeltaP/Q)/(DeltaP/Q)_ref]',
        'defensible as viscosity?': 'Yes, with fixed hydraulic path',
        'scientific basis / limitation': 'For laminar fixed-geometry flow, DeltaP/Q is proportional to dynamic viscosity. The rho_ref/rho factor converts to kinematic viscosity.',
    },
    {
        'proxy': 'Hydraulic power',
        '0422 room 40% anchor': f'P_h={phyd_ref:.3f} W; P_h/Q^2={phyd_q2_ref:.6f} W/(L/min)^2',
        'scaling shown': 'nu_hat = nu_25 * (rho_ref/rho) * [(P_h/Q^2)/(P_h/Q^2)_ref]',
        'defensible as viscosity?': 'Yes, cross-check',
        'scientific basis / limitation': 'P_h=1.6667*DeltaP_bar*Q_L/min, so P_h/Q^2 is the same hydraulic resistance information as DeltaP/Q.',
    },
    {
        'proxy': 'Pump inlet pressure spread',
        '0422 room 40% anchor': f'std_120s(P_in)={pin_spread_ref:.5f} bar',
        'scaling shown': 'index only: nu_25 * std_120s(P_in)/std_120s(P_in)_ref',
        'defensible as viscosity?': 'Qualitative only',
        'scientific basis / limitation': 'Pressure spread can reflect pulsation, vapor, control quantization, or cavitation margin. It can be compared to viscosity trends, but is not a viscosity law.',
    },
    {
        'proxy': 'Input energy',
        '0422 room 40% anchor': f'(60*P_e/Q)={ein_ref:.1f} J/L',
        'scaling shown': 'index only: nu_25 * [(60*P_e/Q)/(60*P_e/Q)_ref]',
        'defensible as viscosity?': 'Qualitative only',
        'scientific basis / limitation': 'Electrical input includes VFD/motor overhead and pump efficiency. It becomes a viscosity estimate only with a separate baseline/efficiency model.',
    },
]).set_index('proxy')

physical_proxy_frame = visc[
    visc['pump_running']
    & visc['pump_cmd_pct'].sub(ROOM_REFERENCE_PUMP_CMD_PCT).abs().le(ROOM_REFERENCE_PUMP_CMD_TOL_PCT)
    & visc['time_s'].ge(BYPASS_CLOSED_TIME_S)
].copy().sort_values('time_s').reset_index(drop=True)
physical_proxy_frame = physical_proxy_frame[
    physical_proxy_frame['bulk_C'].between(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
].copy()
physical_proxy_frame = _add_hydraulic_proxy_primitives(physical_proxy_frame)
physical_proxy_frame['hydraulic_resistance_120s_median'] = _rolling_time_stat(
    physical_proxy_frame,
    'hydraulic_resistance_bar_per_lmin',
    PHYSICAL_PROXY_SMOOTHING_WINDOW_S,
    agg='median',
    min_periods=10,
)
physical_proxy_frame['hydraulic_power_q2_120s_median'] = _rolling_time_stat(
    physical_proxy_frame,
    'hydraulic_power_per_flow2_w_per_lmin2',
    PHYSICAL_PROXY_SMOOTHING_WINDOW_S,
    agg='median',
    min_periods=10,
)
physical_proxy_frame['pump_inlet_pressure_spread_120s_bar'] = _rolling_time_stat(
    physical_proxy_frame,
    'pump_pressure_before_bar_abs',
    ROOM_REFERENCE_PIN_SPREAD_WINDOW_S,
    agg='std',
    min_periods=10,
)
physical_proxy_frame['specific_input_energy_120s_median'] = _rolling_time_stat(
    physical_proxy_frame,
    'specific_input_energy_j_l',
    PHYSICAL_PROXY_SMOOTHING_WINDOW_S,
    agg='median',
    min_periods=10,
)

rho_current = physical_proxy_frame['density_kg_m3_si'].replace(0.0, np.nan)
physical_proxy_frame['density_index_cSt'] = HFE7200_ROOM_KINEMATIC_VISCOSITY_CST * physical_proxy_frame['density_kg_m3_si'] / rho_ref
physical_proxy_frame['pressure_rise_viscosity_cSt'] = (
    HFE7200_ROOM_KINEMATIC_VISCOSITY_CST
    * (rho_ref / rho_current)
    * (physical_proxy_frame['hydraulic_resistance_120s_median'] / rhyd_ref)
)
physical_proxy_frame['hydraulic_power_viscosity_cSt'] = (
    HFE7200_ROOM_KINEMATIC_VISCOSITY_CST
    * (rho_ref / rho_current)
    * (physical_proxy_frame['hydraulic_power_q2_120s_median'] / phyd_q2_ref)
)
physical_proxy_frame['pump_inlet_pressure_spread_index_cSt'] = (
    HFE7200_ROOM_KINEMATIC_VISCOSITY_CST
    * physical_proxy_frame['pump_inlet_pressure_spread_120s_bar']
    / pin_spread_ref
)
physical_proxy_frame['input_energy_index_cSt'] = (
    HFE7200_ROOM_KINEMATIC_VISCOSITY_CST
    * physical_proxy_frame['specific_input_energy_120s_median']
    / ein_ref
)


def _summarize_room_anchor_proxy(frame, column, label, status):
    values = pd.to_numeric(frame[column], errors='coerce')
    valid = values.notna() & values.gt(0.0) & frame['bulk_C'].notna()
    proxy_values = frame.loc[valid, ['bulk_C']].copy()
    proxy_values['value'] = values.loc[valid]
    proxy_values['bin_C'] = pd.cut(
        proxy_values['bulk_C'],
        bins=np.arange(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C + VISCOSITY_PROXY_BIN_WIDTH_C, VISCOSITY_PROXY_BIN_WIDTH_C),
        include_lowest=True,
    )
    summary = proxy_values.groupby('bin_C', observed=True).agg(
        samples=('value', 'size'),
        temp_C=('bulk_C', 'median'),
        median_cSt=('value', 'median'),
        q16_cSt=('value', lambda series: series.quantile(0.16)),
        q84_cSt=('value', lambda series: series.quantile(0.84)),
    )
    summary = summary[summary['samples'].ge(25)].reset_index(drop=True)
    summary['proxy'] = label
    summary['status'] = status
    return summary

room_anchor_proxy_summaries = pd.concat([
    _summarize_room_anchor_proxy(physical_proxy_frame, 'density_index_cSt', 'Density index', 'diagnostic index'),
    _summarize_room_anchor_proxy(physical_proxy_frame, 'pressure_rise_viscosity_cSt', 'Pump pressure rise', 'viscosity estimate'),
    _summarize_room_anchor_proxy(physical_proxy_frame, 'hydraulic_power_viscosity_cSt', 'Hydraulic power', 'viscosity estimate'),
    _summarize_room_anchor_proxy(physical_proxy_frame, 'pump_inlet_pressure_spread_index_cSt', 'Pump inlet pressure spread', 'diagnostic index'),
    _summarize_room_anchor_proxy(physical_proxy_frame, 'input_energy_index_cSt', 'Input energy', 'diagnostic index'),
], ignore_index=True)

anchor_summary_table = pd.DataFrame([
    {'quantity': 'reference log', 'value': ROOM_REFERENCE_LOG_PATH.name},
    {'quantity': 'reference samples', 'value': f'{room_reference_samples} samples'},
    {'quantity': 'reference time span', 'value': f'{room_reference_time_min["min"]:.2f}-{room_reference_time_min["max"]:.2f} min'},
    {'quantity': 'flow/bulk temperature', 'value': f'{room_temp_ref:.2f} C / {bulk_temp_ref:.2f} C'},
    {'quantity': 'pump command/frequency', 'value': f'{ROOM_REFERENCE_PUMP_CMD_PCT:.0f}% / {pump_freq_ref:.2f} Hz'},
    {'quantity': '3M room viscosity used', 'value': f'{HFE7200_ROOM_KINEMATIC_VISCOSITY_CST:.2f} cSt at {HFE7200_ROOM_TEMP_C:.0f} C'},
]).set_index('quantity')

display(Markdown(
    f"Room anchor from `{ROOM_REFERENCE_LOG_PATH.name}`: `{room_reference_samples}` samples at `{ROOM_REFERENCE_PUMP_CMD_PCT:.0f}%` pump command, "
    f"time `{room_reference_time_min['min']:.2f}-{room_reference_time_min['max']:.2f} min`, "
    f"median flow temperature `{room_temp_ref:.2f} C` and bulk temperature `{bulk_temp_ref:.2f} C`. "
    f"The physical estimates below assume this 0422 segment and the 0424 post-bypass segment have the same hydraulic path."
))
display(anchor_summary_table)
display(proxy_defensibility_table)

fig, ax = plt.subplots(figsize=(10.5, 6.0))
reference_plot = reference_viscosity[
    reference_viscosity['temperature_c'].between(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
]
ax.plot(
    reference_plot['temperature_c'],
    reference_plot['kinematic_viscosity_cSt'],
    color='black',
    lw=1.8,
    marker='o',
    ms=4,
    label='3M HFE-7200 data (comparison only)',
)
proxy_styles = {
    'Density index': {'color': '#7a5195', 'ls': ':', 'alpha': 0.55},
    'Pump pressure rise': {'color': '#ef5675', 'ls': '-', 'alpha': 0.18},
    'Hydraulic power': {'color': '#ffa600', 'ls': '-', 'alpha': 0.18},
    'Pump inlet pressure spread': {'color': '#2f4b7c', 'ls': '--', 'alpha': 0.12},
    'Input energy': {'color': '#00876c', 'ls': '--', 'alpha': 0.12},
}
for label, summary in room_anchor_proxy_summaries.groupby('proxy', sort=False):
    style = proxy_styles[label]
    summary = summary.sort_values('temp_C')
    ax.plot(summary['temp_C'], summary['median_cSt'], color=style['color'], lw=1.6, ls=style['ls'], label=label)
    ax.fill_between(
        summary['temp_C'].to_numpy(float),
        summary['q16_cSt'].to_numpy(float),
        summary['q84_cSt'].to_numpy(float),
        color=style['color'],
        alpha=style['alpha'],
        lw=0,
    )
ax.axhline(HFE7200_ROOM_KINEMATIC_VISCOSITY_CST, color='0.35', lw=1.0, ls=':', alpha=0.75)
ax.set_xlim(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
ax.set_yscale('log')
ax.set_ylim(0.15, 120.0)
ax.set_xlabel('Bulk fluid temperature [C]')
ax.set_ylabel('Room-anchor scaled value [cSt]')
ax.set_title('Room-anchor proxy scaling using only 3M nu_25 and 0422 40% anchors')
ax.legend(loc='upper left', ncols=1, fontsize=8)
ax.text(
    0.98,
    0.04,
    'Physical viscosity estimates:\n'
    'nu = nu_25*(rho_ref/rho)*(DeltaP/Q)/(DeltaP/Q)_ref\n'
    'nu = nu_25*(rho_ref/rho)*(P_h/Q^2)/(P_h/Q^2)_ref\n'
    'Density, spread, and input energy are normalized indices only.',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=7.5,
    bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.92},
)
fig.tight_layout()
plt.show()

room_anchor_proxy_summaries.pivot_table(
    index='proxy',
    values=['temp_C', 'median_cSt', 'q16_cSt', 'q84_cSt'],
    aggfunc={'temp_C': ['min', 'max'], 'median_cSt': ['min', 'max'], 'q16_cSt': 'min', 'q84_cSt': 'max'},
).round(3)


Room anchor from `log_20260422_143345.csv`: `107` samples at `40%` pump command, time `264.77-268.45 min`, median flow temperature `20.90 C` and bulk temperature `20.67 C`. The physical estimates below assume this 0422 segment and the 0424 post-bypass segment have the same hydraulic path.

,value
quantity,
reference log,log_20260422_143345.csv
reference samples,107 samples
reference time span,264.77-268.45 min
flow/bulk temperature,20.90 C / 20.67 C
pump command/frequency,40% / 28.15 Hz
3M room viscosity used,0.41 cSt at 25 C


,0422 room 40% anchor,scaling shown,defensible as viscosity?,scientific basis / limitation
proxy,,,,
Density,1434.93 kg/m3,index only: nu_25 * rho/rho_ref,No,Density is a state/temperature proxy. It is ne...
Pump pressure rise,DeltaP=0.318 bar; DeltaP/Q=0.127325 bar/(L/min),nu_hat = nu_25 * (rho_ref/rho) * [(DeltaP/Q)/(...,"Yes, with fixed hydraulic path","For laminar fixed-geometry flow, DeltaP/Q is p..."
Hydraulic power,P_h=1.360 W; P_h/Q^2=0.212208 W/(L/min)^2,nu_hat = nu_25 * (rho_ref/rho) * [(P_h/Q^2)/(P...,"Yes, cross-check","P_h=1.6667*DeltaP_bar*Q_L/min, so P_h/Q^2 is t..."
Pump inlet pressure spread,std_120s(P_in)=0.05259 bar,index only: nu_25 * std_120s(P_in)/std_120s(P_...,Qualitative only,"Pressure spread can reflect pulsation, vapor, ..."
Input energy,(60*P_e/Q)=1638.8 J/L,index only: nu_25 * [(60*P_e/Q)/(60*P_e/Q)_ref],Qualitative only,Electrical input includes VFD/motor overhead a...


median_cSt        q16_cSt q84_cSt  temp_C         
                                  max    min     min     max     max      min
proxy                                                                        
Density index                   0.477  0.440   0.439   0.478 -28.859 -101.443
Hydraulic power                 1.145  0.478   0.465   1.247 -28.859 -101.443
Input energy                    0.395  0.375   0.372   0.406 -28.859 -101.443
Pump inlet pressure spread      1.813  0.537   0.493   2.163 -28.859 -101.443
Pump pressure rise              1.145  0.478   0.465   1.247 -28.859 -101.443

## Empirical 3M-Benchmark Viscosity Proxy Curves

This plot is an empirical benchmark, not the physically anchored estimate. It converts each fixed-`40%` pump-command proxy into an implied HFE-7200 kinematic viscosity using a deliberately simple calibration against the full 3M curve: after trying a small grid of centered rolling means/medians, each proxy gets one affine scale into `log(nu)` and is compared against the 3M HFE-7200 viscosity measurements from `HFE Freezing Data.pdf` over `-120 C` to `0 C`. The shaded bands are the 16-84 percent sample spread inside temperature bins after that simple scaling; proxy curves appear only where this run has logged `40%` samples.

In [11]:
VISCOSITY_PROXY_TEMP_MIN_C = -120.0
VISCOSITY_PROXY_TEMP_MAX_C = 0.0
VISCOSITY_PROXY_BIN_WIDTH_C = 2.5
VISCOSITY_PROXY_PUMP_CMD_PCT = 40.0
VISCOSITY_PROXY_PUMP_CMD_TOL_PCT = 0.25
HFE7200_ROOM_TEMP_C = 25.0
HFE7200_ROOM_KINEMATIC_VISCOSITY_CST = float(
    reference_viscosity.loc[
        reference_viscosity['temperature_c'].eq(HFE7200_ROOM_TEMP_C),
        'kinematic_viscosity_cSt',
    ].iloc[0]
)
viscosity_proxy_command_mask = visc['pump_cmd_pct'].sub(VISCOSITY_PROXY_PUMP_CMD_PCT).abs().le(VISCOSITY_PROXY_PUMP_CMD_TOL_PCT)

calibration_frame = visc[
    visc['pump_running']
    & viscosity_proxy_command_mask
    & visc['time_s'].ge(BYPASS_CLOSED_TIME_S)
].copy().sort_values('time_s').reset_index(drop=True)
calibration_frame = calibration_frame[
    calibration_frame['bulk_C'].between(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
    & calibration_frame['nu_hfe7200_ref_cSt'].notna()
].copy()
viscosity_estimate_cmd_min = float(calibration_frame['pump_cmd_pct'].min())
viscosity_estimate_cmd_max = float(calibration_frame['pump_cmd_pct'].max())
viscosity_estimate_freq_median = float(calibration_frame['pump_freq_hz'].median())
display(Markdown(
    f"Viscosity proxy estimates use only pump-command samples within "
    f"`{VISCOSITY_PROXY_PUMP_CMD_PCT:.2f} +/- {VISCOSITY_PROXY_PUMP_CMD_TOL_PCT:.2f}%`: "
    f"`{len(calibration_frame)}` samples, command range "
    f"`{viscosity_estimate_cmd_min:.2f}-{viscosity_estimate_cmd_max:.2f}%`, "
    f"median pump frequency `{viscosity_estimate_freq_median:.2f} Hz`, "
    f"after bypass closed at `{BYPASS_CLOSED_TIME_MIN:.2f} min`. "
    f"This empirical benchmark uses the full 3M HFE-7200 viscosity curve to choose each proxy scale, "
    f"while still reporting estimates in units anchored to the 3M/TDS room-temperature value "
    f"`nu({HFE7200_ROOM_TEMP_C:.0f} C) = {HFE7200_ROOM_KINEMATIC_VISCOSITY_CST:.2f} cSt`."
))


def _rolling_time_values(frame, values, window_s, *, agg='mean', min_periods=None):
    values = pd.Series(np.asarray(values, dtype=float), index=pd.to_timedelta(frame['time_s'].to_numpy(float), unit='s'))
    if window_s <= 0:
        return values.to_numpy()
    if min_periods is None:
        min_periods = max(5, int(round(window_s / 8.0)))
    rolled = values.rolling(f'{float(window_s):.0f}s', center=True, min_periods=min_periods)
    if agg == 'mean':
        return rolled.mean().to_numpy()
    if agg == 'median':
        return rolled.median().to_numpy()
    if agg == 'std':
        return rolled.std().to_numpy()
    raise ValueError(f'Unknown rolling aggregation {agg!r}.')


def _fit_simple_log_viscosity_scale(frame, feature_values, *, proxy_key, proxy_label, feature_label, smoothing_label):
    fit_frame = frame[['time_s', 'bulk_C', 'nu_hfe7200_ref_cSt']].copy()
    fit_frame['feature'] = np.asarray(feature_values, dtype=float)
    valid = (
        fit_frame['feature'].notna()
        & fit_frame['nu_hfe7200_ref_cSt'].notna()
        & np.isfinite(fit_frame['feature'].to_numpy(float))
        & np.isfinite(fit_frame['nu_hfe7200_ref_cSt'].to_numpy(float))
    )
    fit_frame = fit_frame.loc[valid].copy()
    if len(fit_frame) < 120:
        return None

    fit_frame['fit_bin_C'] = pd.cut(
        fit_frame['bulk_C'],
        bins=np.arange(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C + VISCOSITY_PROXY_BIN_WIDTH_C, VISCOSITY_PROXY_BIN_WIDTH_C),
        include_lowest=True,
    )
    binned = fit_frame.groupby('fit_bin_C', observed=True).agg(
        samples=('time_s', 'size'),
        temp_C=('bulk_C', 'median'),
        feature=('feature', 'median'),
        ref_nu_cSt=('nu_hfe7200_ref_cSt', 'median'),
    )
    binned = binned[binned['samples'].ge(25)].dropna()
    if len(binned) < 6 or float(binned['feature'].std()) <= 0.0:
        return None

    feature_center = float(binned['feature'].median())
    feature_scale = float(binned['feature'].quantile(0.84) - binned['feature'].quantile(0.16))
    if not np.isfinite(feature_scale) or abs(feature_scale) < 1e-12:
        feature_scale = float(binned['feature'].std())
    if not np.isfinite(feature_scale) or abs(feature_scale) < 1e-12:
        return None

    z = (binned['feature'].to_numpy(float) - feature_center) / feature_scale
    y = np.log(binned['ref_nu_cSt'].to_numpy(float) / HFE7200_ROOM_KINEMATIC_VISCOSITY_CST)
    slope, intercept_room_scaled = np.polyfit(z, y, 1)
    direct_slope_per_feature = float(slope / feature_scale)
    direct_intercept_room_scaled = float(intercept_room_scaled - direct_slope_per_feature * feature_center)
    pred = intercept_room_scaled + slope * z
    rmse_log = float(np.sqrt(np.mean((pred - y) ** 2)))
    ss_res = float(np.sum((pred - y) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0.0 else np.nan

    sample_z = (fit_frame['feature'].to_numpy(float) - feature_center) / feature_scale
    fit_frame['nu_est_cSt'] = np.clip(
        HFE7200_ROOM_KINEMATIC_VISCOSITY_CST * np.exp(intercept_room_scaled + slope * sample_z),
        0.15,
        250.0,
    )
    fit_frame['proxy_key'] = proxy_key
    fit_frame['proxy_label'] = proxy_label
    fit_frame['feature_label'] = feature_label
    fit_frame['smoothing_label'] = smoothing_label

    return {
        'proxy_key': proxy_key,
        'proxy_label': proxy_label,
        'feature_label': feature_label,
        'smoothing_label': smoothing_label,
        'samples': int(len(fit_frame)),
        'fit_bins': int(len(binned)),
        'room_temp_scale_cSt': HFE7200_ROOM_KINEMATIC_VISCOSITY_CST,
        'intercept_log_nu_over_room': float(intercept_room_scaled),
        'direct_intercept_log_nu_over_room': direct_intercept_room_scaled,
        'direct_slope_log_nu_per_feature': direct_slope_per_feature,
        'intercept_log_nu': float(np.log(HFE7200_ROOM_KINEMATIC_VISCOSITY_CST) + intercept_room_scaled),
        'slope_log_nu_per_scaled_feature': float(slope),
        'rmse_log_cSt': rmse_log,
        'typical_multiplicative_error_pct': 100.0 * (np.exp(rmse_log) - 1.0),
        'r_squared_binned_log': float(r_squared),
        'feature_center': feature_center,
        'feature_scale_16_84': feature_scale,
        'estimate_frame': fit_frame,
    }


smooth_options = [
    (0, 'raw'),
    (30, '30 s rolling mean'),
    (60, '60 s rolling mean'),
    (120, '120 s rolling mean'),
    (240, '240 s rolling mean'),
    (60, '60 s rolling median'),
    (120, '120 s rolling median'),
    (240, '240 s rolling median'),
]
regular_proxy_specs = [
    ('density', 'Density proxy', 'density_kg_m3_si', 'Density [kg/m3]'),
    ('pressure_rise', 'Pump pressure rise proxy', 'delta_p_bar_recomputed', 'Pump pressure rise [bar]'),
    ('hydraulic_power', 'Hydraulic power proxy', 'hydraulic_power_w', 'Hydraulic power [W]'),
    ('input_energy', 'Input energy proxy', 'specific_input_energy_j_l', 'Specific input energy [J/L]'),
]

all_candidates = []
selected_fits = []
for proxy_key, proxy_label, column, feature_label in regular_proxy_specs:
    proxy_candidates = []
    raw_values = calibration_frame[column].to_numpy(float)
    for window_s, smoothing_label in smooth_options:
        if smoothing_label.endswith('median'):
            agg = 'median'
        elif smoothing_label == 'raw':
            agg = 'mean'
        else:
            agg = 'mean'
        feature_values = _rolling_time_values(calibration_frame, raw_values, window_s, agg=agg)
        result = _fit_simple_log_viscosity_scale(
            calibration_frame,
            feature_values,
            proxy_key=proxy_key,
            proxy_label=proxy_label,
            feature_label=feature_label,
            smoothing_label=smoothing_label,
        )
        if result is not None:
            proxy_candidates.append(result)
            all_candidates.append(result)
    selected_fits.append(min(proxy_candidates, key=lambda item: item['rmse_log_cSt']))

# Pump-inlet spread is itself a rolling quantity. Try several spread windows, then optionally smooth it.
inlet_spread_candidates = []
pressure_values = calibration_frame['pump_pressure_before_bar_abs'].to_numpy(float)
for spread_window_s in [30, 60, 120, 240, 360]:
    spread_values = _rolling_time_values(calibration_frame, pressure_values, spread_window_s, agg='std')
    for smooth_window_s in [0, 60, 120, 240]:
        if smooth_window_s > 0:
            feature_values = _rolling_time_values(
                calibration_frame,
                spread_values,
                smooth_window_s,
                agg='mean',
                min_periods=max(5, int(round(smooth_window_s / 10.0))),
            )
            smoothing_label = f'{spread_window_s} s rolling std + {smooth_window_s} s mean'
        else:
            feature_values = spread_values
            smoothing_label = f'{spread_window_s} s rolling std'
        result = _fit_simple_log_viscosity_scale(
            calibration_frame,
            feature_values,
            proxy_key='inlet_spread',
            proxy_label='Pump inlet pressure spread proxy',
            feature_label='Pump inlet pressure spread [bar] ',
            smoothing_label=smoothing_label,
        )
        if result is not None:
            inlet_spread_candidates.append(result)
            all_candidates.append(result)
selected_fits.append(min(inlet_spread_candidates, key=lambda item: item['rmse_log_cSt']))

proxy_fit_summary = pd.DataFrame([
    {key: value for key, value in fit.items() if key != 'estimate_frame'}
    for fit in selected_fits
]).set_index('proxy_label').sort_values('rmse_log_cSt')

display(proxy_fit_summary[[
    'feature_label',
    'smoothing_label',
    'fit_bins',
    'room_temp_scale_cSt',
    'direct_intercept_log_nu_over_room',
    'direct_slope_log_nu_per_feature',
    'feature_center',
    'feature_scale_16_84',
    'rmse_log_cSt',
    'typical_multiplicative_error_pct',
    'r_squared_binned_log',
]].round(4))

candidate_summary = pd.DataFrame([
    {key: value for key, value in fit.items() if key != 'estimate_frame'}
    for fit in all_candidates
])
candidate_summary = candidate_summary.sort_values(['proxy_label', 'rmse_log_cSt']).groupby('proxy_label', observed=True).head(3)
display(candidate_summary[[
    'proxy_label',
    'smoothing_label',
    'rmse_log_cSt',
    'typical_multiplicative_error_pct',
    'r_squared_binned_log',
]].round(4))

estimate_frames = [fit['estimate_frame'] for fit in selected_fits]
proxy_estimates = pd.concat(estimate_frames, ignore_index=True)
proxy_estimates['plot_bin_C'] = pd.cut(
    proxy_estimates['bulk_C'],
    bins=np.arange(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C + VISCOSITY_PROXY_BIN_WIDTH_C, VISCOSITY_PROXY_BIN_WIDTH_C),
    include_lowest=True,
)
proxy_bands = proxy_estimates.groupby(['proxy_key', 'proxy_label', 'plot_bin_C'], observed=True).agg(
    samples=('time_s', 'size'),
    temp_C=('bulk_C', 'median'),
    nu_median_cSt=('nu_est_cSt', 'median'),
    nu_p16_cSt=('nu_est_cSt', lambda s: s.quantile(0.16)),
    nu_p84_cSt=('nu_est_cSt', lambda s: s.quantile(0.84)),
    ref_nu_cSt=('nu_hfe7200_ref_cSt', 'median'),
)
proxy_bands = proxy_bands[proxy_bands['samples'].ge(20)].reset_index()

proxy_colors = {
    'density': '#2563eb',
    'pressure_rise': '#dc2626',
    'hydraulic_power': '#7c3aed',
    'inlet_spread': '#0891b2',
    'input_energy': '#ea580c',
}
proxy_plot_order = [
    'density',
    'pressure_rise',
    'hydraulic_power',
    'inlet_spread',
    'input_energy',
]

fit_by_proxy_key = {fit['proxy_key']: fit for fit in selected_fits}
proxy_feature_symbols = {
    'density': 'rho',
    'pressure_rise': 'dP',
    'hydraulic_power': 'dP*Q',
    'inlet_spread': 'std(P_in)',
    'input_energy': 'E_in/L',
}

fig = plt.figure(figsize=(16.4, 8.0), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[3.4, 1.35])
ax = fig.add_subplot(grid[0, 0])
ax_equations = fig.add_subplot(grid[0, 1])
ax_equations.axis('off')
reference_temp_grid = np.linspace(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C, 600)
ax.plot(
    reference_temp_grid,
    hfe7200_kinematic_viscosity_cst(reference_temp_grid),
    color='black',
    lw=2.2,
    label='3M HFE-7200 interpolation',
)
reference_cold_plot = reference_viscosity[reference_viscosity['temperature_c'].between(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)]
ax.scatter(
    reference_cold_plot['temperature_c'],
    reference_cold_plot['kinematic_viscosity_cSt'],
    s=72,
    marker='s',
    color='black',
    edgecolor='white',
    linewidth=0.9,
    zorder=5,
    label='3M HFE Freezing Data points',
)

for proxy_key in proxy_plot_order:
    band = proxy_bands[proxy_bands['proxy_key'].eq(proxy_key)].sort_values('temp_C')
    if band.empty:
        continue
    label = str(band['proxy_label'].iloc[0]).replace(' proxy', '')
    color = proxy_colors[proxy_key]
    ax.fill_between(
        band['temp_C'].to_numpy(float),
        band['nu_p16_cSt'].to_numpy(float),
        band['nu_p84_cSt'].to_numpy(float),
        color=color,
        alpha=0.16,
        linewidth=0,
    )
    ax.plot(
        band['temp_C'],
        band['nu_median_cSt'],
        color=color,
        lw=2.0,
        marker='o',
        ms=4.2,
        label=f'{label} (16-84%)',
    )

ax.set_yscale('log')
ax.set_xlim(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
ax.set_ylim(0.45, 90.0)
ax.set_xlabel('TC mean bulk temperature [C]')
ax.set_ylabel('Kinematic viscosity estimate [cSt]')
ax.set_title('Simple scaled viscosity proxies versus 3M HFE-7200 data')
ax.legend(loc='upper right', fontsize=8.4, ncols=2)
_apply_fine_grid(ax)

equation_lines = [
    'Simple proxy scaling',
    f'Axis: {VISCOSITY_PROXY_TEMP_MIN_C:.0f} to {VISCOSITY_PROXY_TEMP_MAX_C:.0f} C',
    f'Filter: pump command {VISCOSITY_PROXY_PUMP_CMD_PCT:.2f} +/- {VISCOSITY_PROXY_PUMP_CMD_TOL_PCT:.2f}%',
    f'After bypass closed: t >= {BYPASS_CLOSED_TIME_MIN:.2f} min',
    f'N = {len(calibration_frame)}; median freq = {viscosity_estimate_freq_median:.2f} Hz',
    f'Room scale: nu_25C = {HFE7200_ROOM_KINEMATIC_VISCOSITY_CST:.2f} cSt',
    'nu_hat = nu_25C * exp(A + B*x_s)  [cSt]',
    '',
]
for proxy_key in proxy_plot_order:
    fit = fit_by_proxy_key[proxy_key]
    proxy_name = fit['proxy_label'].replace(' proxy', '')
    symbol = proxy_feature_symbols[proxy_key]
    equation_lines.extend([
        proxy_name,
        f"  x_s = {symbol}; {fit['smoothing_label']}",
        f"  A = {fit['direct_intercept_log_nu_over_room']:.3g}",
        f"  B = {fit['direct_slope_log_nu_per_feature']:.3g} per x_s",
        '',
    ])
ax_equations.text(
    0.0,
    1.0,
    '\n'.join(equation_lines).rstrip(),
    va='top',
    ha='left',
    fontsize=7.6,
    family='monospace',
    linespacing=1.18,
    bbox=dict(facecolor='white', edgecolor='0.75', boxstyle='round,pad=0.45', alpha=0.96),
)
plt.show()

comparison_rows = []
for proxy_key in proxy_plot_order:
    band = proxy_bands[proxy_bands['proxy_key'].eq(proxy_key)].copy()
    if band.empty:
        continue
    log_error = np.log(band['nu_median_cSt'].to_numpy(float)) - np.log(band['ref_nu_cSt'].to_numpy(float))
    comparison_rows.append({
        'proxy': str(band['proxy_label'].iloc[0]),
        'median_abs_log_error': float(np.median(np.abs(log_error))),
        'median_abs_pct_error': float(100.0 * np.median(np.abs(np.exp(log_error) - 1.0))),
        'max_abs_pct_error': float(100.0 * np.max(np.abs(np.exp(log_error) - 1.0))),
        'median_band_half_width_pct': float(50.0 * np.median((band['nu_p84_cSt'] - band['nu_p16_cSt']) / band['nu_median_cSt'])),
    })
proxy_plot_comparison = pd.DataFrame(comparison_rows).set_index('proxy')
display(proxy_plot_comparison.round(3))

proxy_rank_vs_3m = proxy_fit_summary[[
    'smoothing_label',
    'rmse_log_cSt',
    'typical_multiplicative_error_pct',
    'r_squared_binned_log',
]].join(proxy_plot_comparison)
proxy_rank_vs_3m['rank_by_fit_rmse'] = proxy_rank_vs_3m['rmse_log_cSt'].rank(method='min').astype(int)
proxy_rank_vs_3m['rank_by_median_abs_pct'] = proxy_rank_vs_3m['median_abs_pct_error'].rank(method='min').astype(int)
proxy_rank_vs_3m = proxy_rank_vs_3m.sort_values(['rank_by_fit_rmse', 'rank_by_median_abs_pct'])
display(proxy_rank_vs_3m[[
    'rank_by_fit_rmse',
    'rank_by_median_abs_pct',
    'smoothing_label',
    'rmse_log_cSt',
    'median_abs_pct_error',
    'r_squared_binned_log',
    'median_band_half_width_pct',
]].round(3))

input_energy_drop_band = calibration_frame[
    calibration_frame['bulk_C'].between(-72.5, -45.0)
].copy()
input_energy_drop_band['diagnostic_bin_C'] = pd.cut(
    input_energy_drop_band['bulk_C'],
    bins=np.arange(-72.5, -42.4, 2.5),
    include_lowest=True,
)
input_energy_drop_diagnostic = input_energy_drop_band.groupby('diagnostic_bin_C', observed=True).agg(
    samples=('time_s', 'size'),
    median_bulk_C=('bulk_C', 'median'),
    pump_input_power_w=('pump_input_power_w', 'median'),
    volume_flow_lmin=('volume_flow_lmin_si', 'median'),
    specific_input_energy_j_l=('specific_input_energy_j_l', 'median'),
    hydraulic_power_w=('hydraulic_power_w', 'median'),
    pump_freq_hz=('pump_freq_hz', 'median'),
)
display(input_energy_drop_diagnostic.round(3))

best_fit = proxy_fit_summary.iloc[0]
display(Markdown(
    f"The best simple scaled proxy in this pass is `{proxy_fit_summary.index[0]}` using "
    f"`{best_fit['smoothing_label']}`. Its binned log-viscosity RMSE is "
    f"`{best_fit['rmse_log_cSt']:.3f}` (about `{best_fit['typical_multiplicative_error_pct']:.1f}%` multiplicative error). "
    f"The pressure-derived proxies are intentionally kept as one-scale calibrations, so their shaded bands show the real remaining scatter rather than a high-order fit."
))

drop_row = input_energy_drop_diagnostic.loc[input_energy_drop_diagnostic['median_bulk_C'].between(-60.0, -57.5)].iloc[0]
before_drop_row = input_energy_drop_diagnostic.loc[input_energy_drop_diagnostic['median_bulk_C'].between(-62.5, -60.0)].iloc[0]
display(Markdown(
    f"Hydraulic power is computed as `delta_p_bar_recomputed * volume_flow_lmin_si * BAR_LMIN_TO_W`, "
    f"where `delta_p_bar_recomputed = pump_pressure_after_bar_abs - pump_pressure_before_bar_abs` "
    f"and `BAR_LMIN_TO_W = {orca_logbook.BAR_LMIN_TO_W:.4f} W/(bar L/min)`. "
    f"Around `-60 C`, the input-energy dip is driven mostly by the electrical input readback: "
    f"the median pump input falls from `{before_drop_row['pump_input_power_w']:.0f} W` "
    f"to `{drop_row['pump_input_power_w']:.0f} W`, while flow changes only from "
    f"`{before_drop_row['volume_flow_lmin']:.3f}` to `{drop_row['volume_flow_lmin']:.3f} L/min`. "
    f"Because `specific_input_energy_j_l = 60 * pump_input_power_w / volume_flow_lmin_si`, "
    f"that one-watt readback step produces the apparent local drop. Pump command and frequency remain fixed, "
    f"so this is more likely VFD/power-readback quantization or motor-load behavior than a true viscosity feature."
))


Viscosity proxy estimates use only pump-command samples within `40.00 +/- 0.25%`: `6389` samples, command range `40.00-40.00%`, median pump frequency `28.14 Hz`, after bypass closed at `57.96 min`. This empirical benchmark uses the full 3M HFE-7200 viscosity curve to choose each proxy scale, while still reporting estimates in units anchored to the 3M/TDS room-temperature value `nu(25 C) = 0.41 cSt`.

,feature_label,smoothing_label,fit_bins,room_temp_scale_cSt,direct_intercept_log_nu_over_room,direct_slope_log_nu_per_feature,feature_center,feature_scale_16_84,rmse_log_cSt,typical_multiplicative_error_pct,r_squared_binned_log
proxy_label,,,,,,,,,,,
Density proxy,Density [kg/m3],60 s rolling mean,30,0.41,-29.4223,0.0196,1608.9310,80.5428,0.1251,13.3273,0.9721
Hydraulic power proxy,Hydraulic power [W],240 s rolling median,30,0.41,-0.1884,0.8311,2.4614,1.4878,0.2205,24.6685,0.9132
Pump pressure rise proxy,Pump pressure rise [bar],240 s rolling median,30,0.41,-0.3081,3.9165,0.5525,0.3152,0.2227,24.9453,0.9115
Pump inlet pressure spread proxy,Pump inlet pressure spread [bar],30 s rolling std,30,0.41,-0.1100,17.0840,0.1203,0.0454,0.4398,55.2394,0.6548
Input energy proxy,Specific input energy [J/L],raw,30,0.41,-25.8524,0.0183,1525.0057,27.1628,0.6848,98.3361,0.1631


,proxy_label,smoothing_label,rmse_log_cSt,typical_multiplicative_error_pct,r_squared_binned_log
2,Density proxy,60 s rolling mean,0.1251,13.3273,0.9721
1,Density proxy,30 s rolling mean,0.1254,13.3607,0.9719
3,Density proxy,120 s rolling mean,0.1255,13.3692,0.9719
23,Hydraulic power proxy,240 s rolling median,0.2205,24.6685,0.9132
21,Hydraulic power proxy,60 s rolling median,0.2251,25.2475,0.9096
22,Hydraulic power proxy,120 s rolling median,0.2280,25.6035,0.9073
24,Input energy proxy,raw,0.6848,98.3361,0.1631
25,Input energy proxy,30 s rolling mean,0.6999,101.3520,0.1259
27,Input energy proxy,120 s rolling mean,0.7002,101.4066,0.1252
32,Pump inlet pressure spread proxy,30 s rolling std,0.4398,55.2394,0.6548


,median_abs_log_error,median_abs_pct_error,max_abs_pct_error,median_band_half_width_pct
proxy,,,,
Density proxy,0.116,11.914,20.260,3.286
Pump pressure rise proxy,0.184,16.840,75.114,3.996
Hydraulic power proxy,0.180,16.794,75.250,4.181
Pump inlet pressure spread proxy,0.298,31.007,89.828,38.710
Input energy proxy,0.530,56.216,212.004,42.973


,rank_by_fit_rmse,rank_by_median_abs_pct,smoothing_label,rmse_log_cSt,median_abs_pct_error,r_squared_binned_log,median_band_half_width_pct
proxy_label,,,,,,,
Density proxy,1,1,60 s rolling mean,0.125,11.914,0.972,3.286
Hydraulic power proxy,2,2,240 s rolling median,0.220,16.794,0.913,4.181
Pump pressure rise proxy,3,3,240 s rolling median,0.223,16.840,0.911,3.996
Pump inlet pressure spread proxy,4,4,30 s rolling std,0.440,31.007,0.655,38.710
Input energy proxy,5,5,raw,0.685,56.216,0.163,42.973


,samples,median_bulk_C,pump_input_power_w,volume_flow_lmin,specific_input_energy_j_l,hydraulic_power_w,pump_freq_hz
diagnostic_bin_C,,,,,,,
"(-72.501, -70.0]",132,-71.215,69.0,2.683,1541.039,2.716,28.14
"(-70.0, -67.5]",132,-68.870,69.0,2.679,1540.127,2.616,28.14
"(-67.5, -65.0]",123,-66.211,68.0,2.673,1527.781,2.397,28.14
"(-65.0, -62.5]",127,-63.733,68.0,2.672,1525.382,2.525,28.14
"(-62.5, -60.0]",135,-61.255,68.0,2.670,1524.629,2.350,28.14
"(-60.0, -57.5]",138,-58.733,67.0,2.663,1514.297,2.260,28.14
"(-57.5, -55.0]",143,-56.251,68.0,2.659,1532.394,2.243,28.14
"(-55.0, -52.5]",150,-53.690,68.0,2.654,1535.197,2.139,28.14
"(-52.5, -50.0]",150,-51.242,67.0,2.650,1515.786,2.080,28.14


The best simple scaled proxy in this pass is `Density proxy` using `60 s rolling mean`. Its binned log-viscosity RMSE is `0.125` (about `13.3%` multiplicative error). The pressure-derived proxies are intentionally kept as one-scale calibrations, so their shaded bands show the real remaining scatter rather than a high-order fit.

Hydraulic power is computed as `delta_p_bar_recomputed * volume_flow_lmin_si * BAR_LMIN_TO_W`, where `delta_p_bar_recomputed = pump_pressure_after_bar_abs - pump_pressure_before_bar_abs` and `BAR_LMIN_TO_W = 1.6667 W/(bar L/min)`. Around `-60 C`, the input-energy dip is driven mostly by the electrical input readback: the median pump input falls from `68 W` to `67 W`, while flow changes only from `2.670` to `2.663 L/min`. Because `specific_input_energy_j_l = 60 * pump_input_power_w / volume_flow_lmin_si`, that one-watt readback step produces the apparent local drop. Pump command and frequency remain fixed, so this is more likely VFD/power-readback quantization or motor-load behavior than a true viscosity feature.

## Seeton-Transform Proxy Calibration

This repeats the proxy calibration with the same Seeton viscosity transform used in `HFE_properties.ipynb`. Instead of fitting `log(nu)`, it fits the Seeton transform of the 3M HFE-7200 viscosity points as a simple linear function of each proxy, then inverts the transform back to cSt. This keeps the scaling simple while using a viscosity transform designed to behave better over a wide temperature range.

This is also a full-3M-curve comparison section. It is useful for judging proxy shape, but the room-anchor section above is the scientifically defensible scaling when only the 3M room-temperature viscosity is allowed.

In [12]:
SEETON_MIN_VISCOSITY_CST = 0.04
SEETON_MAX_VISCOSITY_CST = 250.0
SEETON_BESSEL_OFFSET_CST = 1.244067


def seeton_viscosity_transform(viscosity_cst):
    viscosity_cst = np.asarray(viscosity_cst, dtype=float)
    correction = np.exp(-viscosity_cst) * k0(viscosity_cst + SEETON_BESSEL_OFFSET_CST)
    return np.log(np.log(viscosity_cst + 0.7 + correction))


seeton_inverse_nu_grid = np.geomspace(SEETON_MIN_VISCOSITY_CST, SEETON_MAX_VISCOSITY_CST, 12000)
seeton_inverse_y_grid = seeton_viscosity_transform(seeton_inverse_nu_grid)
seeton_inverse_order = np.argsort(seeton_inverse_y_grid)
seeton_inverse_y_grid = seeton_inverse_y_grid[seeton_inverse_order]
seeton_inverse_nu_grid = seeton_inverse_nu_grid[seeton_inverse_order]
seeton_inverse_log_nu_interp = PchipInterpolator(
    seeton_inverse_y_grid,
    np.log(seeton_inverse_nu_grid),
    extrapolate=True,
)


def inverse_seeton_viscosity_transform(transform_value):
    transform_value = np.asarray(transform_value, dtype=float)
    clipped = np.clip(transform_value, float(seeton_inverse_y_grid.min()), float(seeton_inverse_y_grid.max()))
    return np.exp(seeton_inverse_log_nu_interp(clipped))


seeton_reference_fit = np.polyfit(
    np.log(reference_viscosity['temperature_c'].to_numpy(float) + 273.15),
    seeton_viscosity_transform(reference_viscosity['kinematic_viscosity_cSt'].to_numpy(float)),
    1,
)
seeton_reference_A = float(seeton_reference_fit[1])
seeton_reference_B = float(-seeton_reference_fit[0])


def _fit_seeton_proxy_scale(frame, feature_values, *, proxy_key, proxy_label, feature_label, smoothing_label):
    fit_frame = frame[['time_s', 'bulk_C', 'nu_hfe7200_ref_cSt']].copy()
    fit_frame['feature'] = np.asarray(feature_values, dtype=float)
    valid = (
        fit_frame['feature'].notna()
        & fit_frame['nu_hfe7200_ref_cSt'].notna()
        & np.isfinite(fit_frame['feature'].to_numpy(float))
        & np.isfinite(fit_frame['nu_hfe7200_ref_cSt'].to_numpy(float))
        & fit_frame['nu_hfe7200_ref_cSt'].gt(0.0)
    )
    fit_frame = fit_frame.loc[valid].copy()
    if len(fit_frame) < 120:
        return None

    fit_frame['fit_bin_C'] = pd.cut(
        fit_frame['bulk_C'],
        bins=np.arange(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C + VISCOSITY_PROXY_BIN_WIDTH_C, VISCOSITY_PROXY_BIN_WIDTH_C),
        include_lowest=True,
    )
    binned = fit_frame.groupby('fit_bin_C', observed=True).agg(
        samples=('time_s', 'size'),
        temp_C=('bulk_C', 'median'),
        feature=('feature', 'median'),
        ref_nu_cSt=('nu_hfe7200_ref_cSt', 'median'),
    )
    binned = binned[binned['samples'].ge(25)].dropna()
    if len(binned) < 6 or float(binned['feature'].std()) <= 0.0:
        return None

    x = binned['feature'].to_numpy(float)
    y = seeton_viscosity_transform(binned['ref_nu_cSt'].to_numpy(float))
    slope, intercept = np.polyfit(x, y, 1)
    pred_transform = intercept + slope * x
    pred_nu = inverse_seeton_viscosity_transform(pred_transform)
    ref_nu = binned['ref_nu_cSt'].to_numpy(float)
    rmse_log = float(np.sqrt(np.mean((np.log(pred_nu) - np.log(ref_nu)) ** 2)))
    ss_res = float(np.sum((pred_transform - y) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0.0 else np.nan

    sample_transform = intercept + slope * fit_frame['feature'].to_numpy(float)
    fit_frame['nu_est_cSt'] = np.clip(
        inverse_seeton_viscosity_transform(sample_transform),
        SEETON_MIN_VISCOSITY_CST,
        SEETON_MAX_VISCOSITY_CST,
    )
    fit_frame['proxy_key'] = proxy_key
    fit_frame['proxy_label'] = proxy_label
    fit_frame['feature_label'] = feature_label
    fit_frame['smoothing_label'] = smoothing_label

    return {
        'proxy_key': proxy_key,
        'proxy_label': proxy_label,
        'feature_label': feature_label,
        'smoothing_label': smoothing_label,
        'samples': int(len(fit_frame)),
        'fit_bins': int(len(binned)),
        'seeton_intercept': float(intercept),
        'seeton_slope_per_feature': float(slope),
        'rmse_log_cSt': rmse_log,
        'typical_multiplicative_error_pct': 100.0 * (np.exp(rmse_log) - 1.0),
        'r_squared_seeton_transform': float(r_squared),
        'feature_median': float(binned['feature'].median()),
        'feature_16_84_span': float(binned['feature'].quantile(0.84) - binned['feature'].quantile(0.16)),
        'estimate_frame': fit_frame,
    }


seeton_candidates = []
seeton_selected_fits = []
for proxy_key, proxy_label, column, feature_label in regular_proxy_specs:
    proxy_candidates = []
    raw_values = calibration_frame[column].to_numpy(float)
    for window_s, smoothing_label in smooth_options:
        agg = 'median' if smoothing_label.endswith('median') else 'mean'
        feature_values = _rolling_time_values(calibration_frame, raw_values, window_s, agg=agg)
        result = _fit_seeton_proxy_scale(
            calibration_frame,
            feature_values,
            proxy_key=proxy_key,
            proxy_label=proxy_label,
            feature_label=feature_label,
            smoothing_label=smoothing_label,
        )
        if result is not None:
            proxy_candidates.append(result)
            seeton_candidates.append(result)
    seeton_selected_fits.append(min(proxy_candidates, key=lambda item: item['rmse_log_cSt']))

seeton_inlet_spread_candidates = []
pressure_values = calibration_frame['pump_pressure_before_bar_abs'].to_numpy(float)
for spread_window_s in [30, 60, 120, 240, 360]:
    spread_values = _rolling_time_values(calibration_frame, pressure_values, spread_window_s, agg='std')
    for smooth_window_s in [0, 60, 120, 240]:
        if smooth_window_s > 0:
            feature_values = _rolling_time_values(
                calibration_frame,
                spread_values,
                smooth_window_s,
                agg='mean',
                min_periods=max(5, int(round(smooth_window_s / 10.0))),
            )
            smoothing_label = f'{spread_window_s} s rolling std + {smooth_window_s} s mean'
        else:
            feature_values = spread_values
            smoothing_label = f'{spread_window_s} s rolling std'
        result = _fit_seeton_proxy_scale(
            calibration_frame,
            feature_values,
            proxy_key='inlet_spread',
            proxy_label='Pump inlet pressure spread proxy',
            feature_label='Pump inlet pressure spread [bar]',
            smoothing_label=smoothing_label,
        )
        if result is not None:
            seeton_inlet_spread_candidates.append(result)
            seeton_candidates.append(result)
seeton_selected_fits.append(min(seeton_inlet_spread_candidates, key=lambda item: item['rmse_log_cSt']))

seeton_proxy_fit_summary = pd.DataFrame([
    {key: value for key, value in fit.items() if key != 'estimate_frame'}
    for fit in seeton_selected_fits
]).set_index('proxy_label').sort_values('rmse_log_cSt')

display(Markdown(
    f"3M reference Seeton fit: `S(nu) = A - B ln(T_K)`, "
    f"`A = {seeton_reference_A:.4f}`, `B = {seeton_reference_B:.4f}`. "
    f"Proxy fits below use `S(nu_hat) = A_proxy + B_proxy * x_s`."
))
display(seeton_proxy_fit_summary[[
    'feature_label',
    'smoothing_label',
    'fit_bins',
    'seeton_intercept',
    'seeton_slope_per_feature',
    'rmse_log_cSt',
    'typical_multiplicative_error_pct',
    'r_squared_seeton_transform',
]].round(4))

seeton_estimate_frames = [fit['estimate_frame'] for fit in seeton_selected_fits]
seeton_proxy_estimates = pd.concat(seeton_estimate_frames, ignore_index=True)
seeton_proxy_estimates['plot_bin_C'] = pd.cut(
    seeton_proxy_estimates['bulk_C'],
    bins=np.arange(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C + VISCOSITY_PROXY_BIN_WIDTH_C, VISCOSITY_PROXY_BIN_WIDTH_C),
    include_lowest=True,
)
seeton_proxy_bands = seeton_proxy_estimates.groupby(['proxy_key', 'proxy_label', 'plot_bin_C'], observed=True).agg(
    samples=('time_s', 'size'),
    temp_C=('bulk_C', 'median'),
    nu_median_cSt=('nu_est_cSt', 'median'),
    nu_p16_cSt=('nu_est_cSt', lambda s: s.quantile(0.16)),
    nu_p84_cSt=('nu_est_cSt', lambda s: s.quantile(0.84)),
    ref_nu_cSt=('nu_hfe7200_ref_cSt', 'median'),
)
seeton_proxy_bands = seeton_proxy_bands[seeton_proxy_bands['samples'].ge(20)].reset_index()

fig = plt.figure(figsize=(16.4, 8.0), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[3.4, 1.35])
ax = fig.add_subplot(grid[0, 0])
ax_equations = fig.add_subplot(grid[0, 1])
ax_equations.axis('off')
ax.plot(
    reference_temp_grid,
    hfe7200_kinematic_viscosity_cst(reference_temp_grid),
    color='black',
    lw=2.2,
    label='3M HFE-7200 interpolation',
)
ax.scatter(
    reference_cold_plot['temperature_c'],
    reference_cold_plot['kinematic_viscosity_cSt'],
    s=72,
    marker='s',
    color='black',
    edgecolor='white',
    linewidth=0.9,
    zorder=5,
    label='3M HFE Freezing Data points',
)
for proxy_key in proxy_plot_order:
    band = seeton_proxy_bands[seeton_proxy_bands['proxy_key'].eq(proxy_key)].sort_values('temp_C')
    if band.empty:
        continue
    label = str(band['proxy_label'].iloc[0]).replace(' proxy', '')
    color = proxy_colors[proxy_key]
    ax.fill_between(
        band['temp_C'].to_numpy(float),
        band['nu_p16_cSt'].to_numpy(float),
        band['nu_p84_cSt'].to_numpy(float),
        color=color,
        alpha=0.16,
        linewidth=0,
    )
    ax.plot(
        band['temp_C'],
        band['nu_median_cSt'],
        color=color,
        lw=2.0,
        marker='o',
        ms=4.2,
        label=f'{label} Seeton proxy',
    )
ax.set_yscale('log')
ax.set_xlim(VISCOSITY_PROXY_TEMP_MIN_C, VISCOSITY_PROXY_TEMP_MAX_C)
ax.set_ylim(0.45, 90.0)
ax.set_xlabel('TC mean bulk temperature [C]')
ax.set_ylabel('Kinematic viscosity estimate [cSt]')
ax.set_title('Seeton-transform viscosity proxies versus 3M HFE-7200 data')
ax.legend(loc='upper right', fontsize=8.4, ncols=2)
_apply_fine_grid(ax)

seeton_fit_by_proxy_key = {fit['proxy_key']: fit for fit in seeton_selected_fits}
equation_lines = [
    'Seeton proxy scaling',
    'S(nu) = ln ln(nu + 0.7 + exp(-nu)K0(nu+1.244))',
    'S(nu_hat) = A + B*x_s',
    f'Filter: 40% pump, t >= {BYPASS_CLOSED_TIME_MIN:.2f} min',
    '',
]
for proxy_key in proxy_plot_order:
    fit = seeton_fit_by_proxy_key[proxy_key]
    proxy_name = fit['proxy_label'].replace(' proxy', '')
    symbol = proxy_feature_symbols[proxy_key]
    equation_lines.extend([
        proxy_name,
        f"  x_s = {symbol}; {fit['smoothing_label']}",
        f"  A = {fit['seeton_intercept']:.4f}",
        f"  B = {fit['seeton_slope_per_feature']:.4g} per x_s",
        '',
    ])
ax_equations.text(
    0.0,
    1.0,
    '\n'.join(equation_lines).rstrip(),
    va='top',
    ha='left',
    fontsize=7.0,
    family='monospace',
    linespacing=1.15,
    bbox=dict(facecolor='white', edgecolor='0.75', boxstyle='round,pad=0.45', alpha=0.96),
)
plt.show()

seeton_comparison_rows = []
for proxy_key in proxy_plot_order:
    band = seeton_proxy_bands[seeton_proxy_bands['proxy_key'].eq(proxy_key)].copy()
    if band.empty:
        continue
    log_error = np.log(band['nu_median_cSt'].to_numpy(float)) - np.log(band['ref_nu_cSt'].to_numpy(float))
    seeton_comparison_rows.append({
        'proxy': str(band['proxy_label'].iloc[0]),
        'median_abs_log_error': float(np.median(np.abs(log_error))),
        'median_abs_pct_error': float(100.0 * np.median(np.abs(np.exp(log_error) - 1.0))),
        'max_abs_pct_error': float(100.0 * np.max(np.abs(np.exp(log_error) - 1.0))),
        'median_band_half_width_pct': float(50.0 * np.median((band['nu_p84_cSt'] - band['nu_p16_cSt']) / band['nu_median_cSt'])),
    })
seeton_proxy_comparison = pd.DataFrame(seeton_comparison_rows).set_index('proxy')
seeton_proxy_rank_vs_3m = seeton_proxy_fit_summary[[
    'smoothing_label',
    'rmse_log_cSt',
    'typical_multiplicative_error_pct',
    'r_squared_seeton_transform',
]].join(seeton_proxy_comparison)
seeton_proxy_rank_vs_3m['rank_by_fit_rmse'] = seeton_proxy_rank_vs_3m['rmse_log_cSt'].rank(method='min').astype(int)
seeton_proxy_rank_vs_3m['rank_by_median_abs_pct'] = seeton_proxy_rank_vs_3m['median_abs_pct_error'].rank(method='min').astype(int)
seeton_proxy_rank_vs_3m = seeton_proxy_rank_vs_3m.sort_values(['rank_by_fit_rmse', 'rank_by_median_abs_pct'])
display(seeton_proxy_rank_vs_3m[[
    'rank_by_fit_rmse',
    'rank_by_median_abs_pct',
    'smoothing_label',
    'rmse_log_cSt',
    'median_abs_pct_error',
    'r_squared_seeton_transform',
    'median_band_half_width_pct',
]].round(3))

log_vs_seeton_rank = proxy_rank_vs_3m[[
    'rmse_log_cSt',
    'median_abs_pct_error',
]].rename(columns={
    'rmse_log_cSt': 'log_scale_rmse_log_cSt',
    'median_abs_pct_error': 'log_scale_median_abs_pct_error',
}).join(
    seeton_proxy_rank_vs_3m[[
        'rmse_log_cSt',
        'median_abs_pct_error',
    ]].rename(columns={
        'rmse_log_cSt': 'seeton_rmse_log_cSt',
        'median_abs_pct_error': 'seeton_median_abs_pct_error',
    })
)
log_vs_seeton_rank['rmse_delta_seeton_minus_log'] = log_vs_seeton_rank['seeton_rmse_log_cSt'] - log_vs_seeton_rank['log_scale_rmse_log_cSt']
display(log_vs_seeton_rank.round(3))


3M reference Seeton fit: `S(nu) = A - B ln(T_K)`, `A = 23.8671`, `B = 4.4370`. Proxy fits below use `S(nu_hat) = A_proxy + B_proxy * x_s`.

,feature_label,smoothing_label,fit_bins,seeton_intercept,seeton_slope_per_feature,rmse_log_cSt,typical_multiplicative_error_pct,r_squared_seeton_transform
proxy_label,,,,,,,,
Density proxy,Density [kg/m3],120 s rolling mean,30,-18.8925,0.0119,0.0705,7.3021,0.9944
Hydraulic power proxy,Hydraulic power [W],240 s rolling median,30,-1.0521,0.4790,0.3888,47.5154,0.8400
Pump pressure rise proxy,Pump pressure rise [bar],240 s rolling median,30,-1.1203,2.2558,0.3901,47.7155,0.8374
Pump inlet pressure spread proxy,Pump inlet pressure spread [bar],30 s rolling std + 120 s mean,30,-0.9782,9.6011,0.5068,65.9938,0.6062
Input energy proxy,Specific input energy [J/L],raw,30,-15.9993,0.0106,0.6878,98.9321,0.1530


,rank_by_fit_rmse,rank_by_median_abs_pct,smoothing_label,rmse_log_cSt,median_abs_pct_error,r_squared_seeton_transform,median_band_half_width_pct
proxy_label,,,,,,,
Density proxy,1,1,120 s rolling mean,0.070,3.687,0.994,3.184
Hydraulic power proxy,2,2,240 s rolling median,0.389,24.268,0.840,3.477
Pump pressure rise proxy,3,3,240 s rolling median,0.390,24.514,0.837,3.394
Pump inlet pressure spread proxy,4,4,30 s rolling std + 120 s mean,0.507,30.346,0.606,12.545
Input energy proxy,5,5,raw,0.688,48.911,0.153,43.458


,log_scale_rmse_log_cSt,log_scale_median_abs_pct_error,seeton_rmse_log_cSt,seeton_median_abs_pct_error,rmse_delta_seeton_minus_log
proxy_label,,,,,
Density proxy,0.125,11.914,0.070,3.687,-0.055
Hydraulic power proxy,0.220,16.794,0.389,24.268,0.168
Pump pressure rise proxy,0.223,16.840,0.390,24.514,0.167
Pump inlet pressure spread proxy,0.440,31.007,0.507,30.346,0.067
Input energy proxy,0.685,56.216,0.688,48.911,0.003


## Pump Performance Near the 40 to 60 Percent Cold Ramp

The flow-meter did not report `-100 C`; its coldest value is around `-94 C`. The TC-mean bulk proxy does reach about `-100 C`, so the analysis below uses the last cold 40 percent interval as the baseline and compares it to the 45/50/55/60 percent ramp that follows. The 60 percent step warms to roughly `-92 C`, so treat the 60/40 comparison as a near-cold ramp rather than a perfectly isothermal test.


In [13]:
pump_perf = visc.copy()
first_ramp_step_id = int(step_windows[(step_windows.index > sixty_step_id - 10) & step_windows['cmd_pct'].eq(45.0)].index[-1])
ramp_steps = step_windows[(step_windows.index >= first_ramp_step_id) & (step_windows.index <= sixty_step_id)].copy()
baseline_40 = step_windows.loc[step_windows.index[step_windows.index < first_ramp_step_id][-1]]

# Use the last nine minutes of the preceding 40 percent step to keep the baseline close to the ramp temperature.
baseline_start_s = max(float(baseline_40['start_s']), float(baseline_40['end_s']) - 9.0 * 60.0)
comparison_windows = [(int(baseline_40.name), '40 baseline', baseline_start_s, float(baseline_40['end_s']))]
for step_id, row in ramp_steps.iterrows():
    comparison_windows.append((int(step_id), f"{row['cmd_pct']:.0f} ramp", float(row['start_s']) + settle_cutoff_s, float(row['end_s'])))

perf_rows = []
for step_id, label, start_s, end_s in comparison_windows:
    subset = pump_perf[pump_perf['time_s'].between(start_s, end_s) & pump_perf['pump_running']].copy()
    if subset.empty:
        continue
    perf_rows.append({
        'step_id': step_id,
        'label': label,
        'cmd_pct': float(subset['pump_cmd_pct'].round().median()),
        'samples': int(len(subset)),
        'start_min': start_s / 60.0,
        'end_min': end_s / 60.0,
        'bulk_median_C': float(subset['bulk_C'].median()),
        'flow_meter_median_C': float(subset['temperature_c_si'].median()),
        'pump_freq_hz': float(subset['pump_freq_hz'].median()),
        'volume_flow_lmin': float(subset['volume_flow_lmin_si'].median()),
        'mass_flow_kgmin': float(subset['mass_flow_kgmin_si'].median()),
        'density_kg_m3': float(subset['density_kg_m3_si'].median()),
        'delta_p_bar': float(subset['delta_p_bar_recomputed'].median()),
        'pump_input_power_w': float(subset['pump_input_power_w'].median()),
        'pump_current_a': float(subset['pump_output_current_a'].median()),
        'pump_inlet_bar_abs': float(subset['pump_pressure_before_bar_abs'].median()),
        'delivered_ml_per_rev': float(subset['delivered_ml_per_rev'].median()),
        'hydraulic_power_w': float(subset['hydraulic_power_w'].median()),
        'specific_input_energy_j_l': float(subset['specific_input_energy_j_l'].median()),
    })
pump_ramp_summary = pd.DataFrame(perf_rows).set_index('label')

display(pump_ramp_summary.round(3))

if '40 baseline' in pump_ramp_summary.index and '60 ramp' in pump_ramp_summary.index:
    ratio_cols = [
        'pump_freq_hz',
        'volume_flow_lmin',
        'mass_flow_kgmin',
        'delta_p_bar',
        'pump_input_power_w',
        'hydraulic_power_w',
        'delivered_ml_per_rev',
        'specific_input_energy_j_l',
    ]
    ramp_ratio = (pump_ramp_summary.loc['60 ramp', ratio_cols] / pump_ramp_summary.loc['40 baseline', ratio_cols]).to_frame('60_over_40')
    display(ramp_ratio.round(3))

fig = plt.figure(figsize=(13.5, 8.5), constrained_layout=True)
grid = fig.add_gridspec(2, 2)
ax_flow = fig.add_subplot(grid[0, 0])
ax_dp = fig.add_subplot(grid[0, 1])
ax_ml = fig.add_subplot(grid[1, 0])
ax_energy = fig.add_subplot(grid[1, 1])

summary = pump_ramp_summary.sort_values('cmd_pct')
colors = ['#2563eb' if cmd == 40 else '#dc2626' if cmd == 60 else '#64748b' for cmd in summary['cmd_pct']]

ax_flow.scatter(summary['pump_freq_hz'], summary['volume_flow_lmin'], s=95, c=colors, edgecolor='black', zorder=3)
for label, row in summary.iterrows():
    ax_flow.annotate(f"{row['cmd_pct']:.0f}%\n{row['bulk_median_C']:.1f} C", (row['pump_freq_hz'], row['volume_flow_lmin']), xytext=(0, 9), textcoords='offset points', ha='center', fontsize=8)
fit = np.polyfit(summary['pump_freq_hz'], summary['volume_flow_lmin'], 1)
x_line = np.linspace(summary['pump_freq_hz'].min() * 0.96, summary['pump_freq_hz'].max() * 1.03, 100)
ax_flow.plot(x_line, np.polyval(fit, x_line), color='0.25', ls='--', lw=1.5)
ax_flow.set_xlabel('Pump frequency [Hz]')
ax_flow.set_ylabel('Volume flow [L/min]')
ax_flow.set_title('Flow remains nearly linear with pump speed')

ax_dp.plot(summary['cmd_pct'], summary['delta_p_bar'], marker='o', color='#dc2626', lw=2.0, label='Pressure rise')
ax_dp_2 = ax_dp.twinx()
ax_dp_2.plot(summary['cmd_pct'], summary['pump_inlet_bar_abs'], marker='s', color='#0f766e', lw=2.0, label='Inlet pressure')
ax_dp.set_xlabel('Pump command [%]')
ax_dp.set_ylabel('Pressure rise [bar]')
ax_dp_2.set_ylabel('Pump inlet [bar abs]')
lines = ax_dp.get_lines() + ax_dp_2.get_lines()
ax_dp.legend(lines, [line.get_label() for line in lines], loc='best', fontsize=8)
ax_dp.set_title('Pressure load rises during the cold ramp')

ax_ml.plot(summary['cmd_pct'], summary['delivered_ml_per_rev'], marker='o', color='#7c3aed', lw=2.0)
ax_ml.set_xlabel('Pump command [%]')
ax_ml.set_ylabel('Delivered volume [mL/rev]')
ax_ml.set_title('Slip proxy is nearly flat')

ax_energy.plot(summary['cmd_pct'], summary['hydraulic_power_w'], marker='o', color='#111827', lw=2.0, label='Hydraulic output')
ax_energy_2 = ax_energy.twinx()
ax_energy_2.plot(summary['cmd_pct'], summary['specific_input_energy_j_l'], marker='s', color='#ea580c', lw=2.0, label='Input energy per liter')
ax_energy.set_xlabel('Pump command [%]')
ax_energy.set_ylabel('Hydraulic power [W]')
ax_energy_2.set_ylabel('Specific input energy [J/L]')
lines = ax_energy.get_lines() + ax_energy_2.get_lines()
ax_energy.legend(lines, [line.get_label() for line in lines], loc='best', fontsize=8)
ax_energy.set_title('Hydraulic output rises faster than electrical input')

for ax in [ax_flow, ax_dp, ax_dp_2, ax_ml, ax_energy, ax_energy_2]:
    _apply_fine_grid(ax)
plt.show()

display(Markdown(
    f"The 60 percent step gives `{pump_ramp_summary.loc['60 ramp', 'volume_flow_lmin']:.2f} L/min` "
    f"versus `{pump_ramp_summary.loc['40 baseline', 'volume_flow_lmin']:.2f} L/min` at the cold 40 percent baseline. "
    f"Delivered volume per revolution changes by "
    f"`{100.0 * (pump_ramp_summary.loc['60 ramp', 'delivered_ml_per_rev'] / pump_ramp_summary.loc['40 baseline', 'delivered_ml_per_rev'] - 1.0):+.1f}%`, "
    f"so the available data do not show a large slip penalty during the ramp."
))


,step_id,cmd_pct,samples,start_min,end_min,bulk_median_C,flow_meter_median_C,pump_freq_hz,volume_flow_lmin,mass_flow_kgmin,density_kg_m3,delta_p_bar,pump_input_power_w,pump_current_a,pump_inlet_bar_abs,delivered_ml_per_rev,hydraulic_power_w,specific_input_energy_j_l
label,,,,,,,,,,,,,,,,,,
40 baseline,6,40.0,259,158.344,167.344,-101.218,-94.213,28.11,2.744,4.591,1673.0,1.231,73.0,2.55,1.891,1.627,5.630,1590.691
45 ramp,7,45.0,12,167.413,167.797,-99.049,-93.932,31.62,3.086,5.159,1672.0,1.423,76.0,2.56,1.852,1.627,7.314,1476.832
50 ramp,8,50.0,31,167.867,168.948,-98.503,-94.167,35.14,3.424,5.725,1672.0,1.691,80.0,2.56,1.647,1.624,9.701,1400.266
55 ramp,9,55.0,17,169.017,169.575,-96.774,-93.366,38.63,3.746,6.254,1670.0,1.887,84.0,2.57,1.520,1.616,11.790,1346.024
60 ramp,10,60.0,90,169.645,172.748,-93.111,-90.105,42.18,4.074,6.773,1663.0,1.882,94.0,2.57,1.569,1.610,12.772,1384.980


,60_over_40
pump_freq_hz,1.501
volume_flow_lmin,1.484
mass_flow_kgmin,1.475
delta_p_bar,1.529
pump_input_power_w,1.288
hydraulic_power_w,2.268
delivered_ml_per_rev,0.989
specific_input_energy_j_l,0.871


The 60 percent step gives `4.07 L/min` versus `2.74 L/min` at the cold 40 percent baseline. Delivered volume per revolution changes by `-1.1%`, so the available data do not show a large slip penalty during the ramp.

## HFE-7200 Properties Versus Temperature

These plots collect the property curves needed to interpret the cold run. Density and vapor pressure use the 3M HFE-7200 equations already carried in the repository; viscosity uses the tabulated low-temperature points from `analysis/docs/HFE Freezing Data.pdf` with a shape-preserving log interpolation. The shaded band marks the TC-mean temperature range reached in this log.


In [14]:
HFE7200_VAPOR_PRESSURE_LOG_INTERCEPT = 22.289
HFE7200_VAPOR_PRESSURE_LOG_SLOPE_K = 3752.1


def hfe7200_density_kg_m3_from_c(temperature_c):
    return 1000.0 * (
        orca_logbook.HFE_7200_DENSITY_INTERCEPT_G_ML
        - orca_logbook.HFE_7200_DENSITY_SLOPE_G_ML_PER_C * np.asarray(temperature_c, dtype=float)
    )


def hfe7200_vapor_pressure_kpa(temperature_c):
    temperature_k_for_fit = np.asarray(temperature_c, dtype=float) + 273.0
    return np.exp(HFE7200_VAPOR_PRESSURE_LOG_INTERCEPT - HFE7200_VAPOR_PRESSURE_LOG_SLOPE_K / temperature_k_for_fit) / 1000.0


def hfe7200_specific_heat_j_kgk_from_c(temperature_c):
    temp_k = np.asarray(temperature_c, dtype=float) + 273.15
    return np.maximum(900.0, 1220.0 + 1.5 * (temp_k - 298.0))

property_t_c = np.linspace(-125.0, 30.0, 500)
property_density = hfe7200_density_kg_m3_from_c(property_t_c)
property_nu = hfe7200_kinematic_viscosity_cst(property_t_c)
property_mu = property_nu * property_density / 1000.0
property_vapor_kpa = hfe7200_vapor_pressure_kpa(property_t_c)
property_cp = hfe7200_specific_heat_j_kgk_from_c(property_t_c)
property_capacity = np.asarray([orca.thermal_capacity_j_per_k(model, temp_c + 273.15) for temp_c in property_t_c])

log_temp_min = float(thermal.loc[thermal['pump_running'], 'bulk_C'].min())
log_temp_max = float(thermal.loc[thermal['pump_running'], 'bulk_C'].max())

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.0), constrained_layout=True)

axes[0, 0].plot(property_t_c, property_density, color='#2563eb', lw=2.1, label='3M density law')
axes[0, 0].scatter(thermal.loc[thermal['pump_running'], 'bulk_C'], thermal.loc[thermal['pump_running'], 'density_kg_m3_si'], s=6, color='#111827', alpha=0.16, label='Logged density')
axes[0, 0].set_ylabel('Density [kg/m3]')
axes[0, 0].set_title('Density')
axes[0, 0].legend(loc='best', fontsize=8)

axes[0, 1].plot(property_t_c, property_nu, color='#dc2626', lw=2.1, label='Kinematic viscosity')
axes[0, 1].plot(property_t_c, property_mu, color='#7c3aed', lw=2.1, label='Dynamic viscosity')
axes[0, 1].scatter(reference_viscosity['temperature_c'], reference_viscosity['kinematic_viscosity_cSt'], color='#dc2626', edgecolor='white', zorder=3, label='Reference nu points')
axes[0, 1].set_yscale('log')
axes[0, 1].set_ylabel('Viscosity [cSt or cP]')
axes[0, 1].set_title('Viscosity')
axes[0, 1].legend(loc='best', fontsize=8)

axes[1, 0].plot(property_t_c, property_vapor_kpa, color='#0f766e', lw=2.1)
axes[1, 0].set_yscale('log')
axes[1, 0].set_ylabel('Vapor pressure [kPa]')
axes[1, 0].set_title('Vapor pressure')

axes[1, 1].plot(property_t_c, property_cp, color='#ea580c', lw=2.1, label='HFE cp model')
axes[1, 1].set_ylabel('Specific heat [J/kg/K]')
axes[1, 1].set_title('Specific heat and total modeled capacity')
axes_cap = axes[1, 1].twinx()
axes_cap.plot(property_t_c, property_capacity / 1000.0, color='#111827', lw=1.8, ls='--', label='Total capacity')
axes_cap.set_ylabel('Total capacity [kJ/K]')
lines = axes[1, 1].get_lines() + axes_cap.get_lines()
axes[1, 1].legend(lines, [line.get_label() for line in lines], loc='best', fontsize=8)

for ax in axes.ravel():
    ax.axvspan(log_temp_min, log_temp_max, color='#60a5fa', alpha=0.10, lw=0, label='Run range')
    ax.axvline(-70.0, color='0.35', ls=':', lw=1.0)
    ax.set_xlim(-125, 30)
    ax.set_xlabel('Temperature [C]')
    _apply_fine_grid(ax)
_apply_fine_grid(axes_cap)
plt.show()

property_samples = pd.DataFrame({
    'temperature_C': [-100.0, -90.0, -70.0, -50.0, 0.0, 25.0],
})
property_samples['density_kg_m3'] = hfe7200_density_kg_m3_from_c(property_samples['temperature_C'])
property_samples['kinematic_viscosity_cSt'] = hfe7200_kinematic_viscosity_cst(property_samples['temperature_C'])
property_samples['dynamic_viscosity_cP'] = property_samples['kinematic_viscosity_cSt'] * property_samples['density_kg_m3'] / 1000.0
property_samples['vapor_pressure_kPa'] = hfe7200_vapor_pressure_kpa(property_samples['temperature_C'])
property_samples['specific_heat_J_kgK'] = hfe7200_specific_heat_j_kgk_from_c(property_samples['temperature_C'])
display(property_samples.round(4))


,temperature_C,density_kg_m3,kinematic_viscosity_cSt,dynamic_viscosity_cP,vapor_pressure_kPa,specific_heat_J_kgK
0,-100.0,1711.360,12.470,21.3407,0.0018,1032.725
1,-90.0,1688.334,7.796,13.1623,0.0060,1047.725
2,-70.0,1642.282,3.720,6.1093,0.0450,1077.725
3,-50.0,1596.230,1.840,2.9371,0.2359,1107.725
4,0.0,1481.100,0.670,0.9923,5.1412,1182.725
5,25.0,1423.535,0.410,0.5836,16.2860,1220.225
